# Pipeline Evaluation: GNN+EBM vs MILP across 3 Criticality Families (LOCAL GPU)

**This is the LOCAL version of the evaluation pipeline notebook**, intended to run on a Windows workstation with an NVIDIA GPU (tested on RTX 4050, 6 GB VRAM, CUDA 12.4).
The Colab version is `evaluation_pipeline.ipynb`.

**Eval Phase Checklist:**
1. Generate 100 x 3 scenarios: easy, medium, hard (based on criticality)
2. Evaluate each family with the full pipeline (Graph -> HTE -> EBM -> Pass-through warm-start -> LP)
3. Compute p90-p99 metrics
4. Build solve time vs cost gap relationship (MILP vs Pipeline)
5. Compute economic advantage indicator: `lambda * time_saved - cost_gap`
6. Sensitivity analysis on lambda (EUR/second)

## Prerequisites (one-time setup)

### System
- Windows 10/11 with an NVIDIA GPU and recent driver (>= 552.x for CUDA 12.4 runtime).
- Python 3.10 or 3.11 (64-bit). Avoid 3.12 (no torch-scatter wheels).
- (Optional) Microsoft Visual C++ Build Tools 2022 — only if a torch-scatter wheel falls back to source build.

### Python environment (PowerShell, from the repo root)
```powershell
py -3.11 -m venv .venv
.\.venv\Scripts\Activate.ps1
python -m pip install --upgrade pip wheel setuptools

# PyTorch 2.5.1 + CUDA 12.4 (Windows wheels)
pip install torch==2.5.1+cu124 torchvision==0.20.1+cu124 torchaudio==2.5.1+cu124 `
    --index-url https://download.pytorch.org/whl/cu124

# PyG + scatter/sparse Windows wheels matching torch 2.5.1+cu124
pip install torch-geometric
pip install torch-scatter -f https://data.pyg.org/whl/torch-2.5.1+cu124.html
pip install torch-sparse  -f https://data.pyg.org/whl/torch-2.5.1+cu124.html

# Scientific + optimisation + plotting stack
pip install numpy "scipy>=1.10" "pandas>=2.0" pyarrow `
    "pyomo>=6.7" "highspy>=1.5" `
    "matplotlib>=3.7" "seaborn>=0.13" `
    hdbscan umap-learn "scikit-learn>=1.3" `
    PyYAML tqdm jupyter ipykernel
```

### Required files on disk (under the repo root)
- `src/` — full source (already in git)
- `outputs/low_criticality_scenarios/` — 100 scenarios + `reports/` (already in repo)
- `outputs/medium_criticality_scenarios/` — 100 scenarios + `reports/`
- `outputs/high_criticality_scenarios/` — 100 scenarios + `reports/`
- `outputs/encoders/hierarchical_temporal_v3/best_encoder.pt` — HTE encoder checkpoint **(download from Google Drive `MyDrive/benchmark/...`)**
- `outputs/ebm_models/ebm_v3/ebm_v3_final.pt` — EBM checkpoint **(download from Google Drive `MyDrive/benchmark/...`)**
- `outputs/rhmo_{low,medium,high}.json` — heuristic baseline results (already in repo)

### RTX 4050 (6 GB VRAM) tips
- The HTE encoder (3.2 M params) + EBM (0.5 M) are small, but high-criticality scenarios (up to 160 zones x 24 timesteps) can stress the temporal-transformer attention.
- If you hit CUDA OOM on the `high` family, lower `n_samples` from 5 to 2-3 and/or `langevin_steps` from 100 to 50 in the `PipelineConfig` cell.

## 1. Verify Local Environment

In [ ]:
# Quick sanity check that all required packages import and CUDA is visible.
# Install them in your venv first - see prerequisites at the top of the notebook.
import sys, importlib

required = [
    'numpy', 'scipy', 'pandas', 'pyarrow', 'tqdm', 'yaml',
    'matplotlib', 'seaborn',
    'torch', 'torch_geometric',
    'pyomo', 'highspy',
]
missing = []
for mod in required:
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(mod)
if missing:
    print(f'MISSING packages: {missing}')
    print('Install them in your venv before continuing (see prerequisites).')
else:
    print('All required packages import OK.')

import torch
print(f'\nPython:   {sys.version.split()[0]}')
print(f'PyTorch:  {torch.__version__}')
print(f'CUDA OK:  {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:      {torch.cuda.get_device_name(0)}')
    print(f'CUDA ver: {torch.version.cuda}')
    print(f'VRAM:     {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Setup Local Paths

In [ ]:
# Clear module cache so updated .py files are reloaded
import importlib, sys

modules_to_clear = [m for m in sys.modules if m.startswith('src.')]
for m in modules_to_clear:
    del sys.modules[m]
print(f"Cleared {len(modules_to_clear)} cached modules")

In [ ]:
import os, sys, json, time, pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any
from tqdm.auto import tqdm

# --- Local repo path ---
# Auto-detect: the notebook lives at <repo>/notebooks/, so the repo is the parent.
NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name == 'notebooks':
    REPO_PATH = NOTEBOOK_DIR.parent
else:
    # Fallback: hard-coded path (edit if your checkout lives elsewhere)
    REPO_PATH = Path(r'C:\Users\coudr\OneDrive\projects\benchmark\benchmark_milp_gnn')

assert REPO_PATH.exists(), f'Repo path not found: {REPO_PATH}'
sys.path.insert(0, str(REPO_PATH))
os.chdir(REPO_PATH)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Repo:   {REPO_PATH}')
print(f'Device: {DEVICE}')
print(f'GPU:    {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

# --- Verify required model checkpoints are present ---
required_files = [
    REPO_PATH / 'outputs/encoders/hierarchical_temporal_v3/best_encoder.pt',
    REPO_PATH / 'outputs/ebm_models/ebm_v3/ebm_v3_final.pt',
]
missing_ckpts = [str(p.relative_to(REPO_PATH)) for p in required_files if not p.exists()]
if missing_ckpts:
    print('\nMISSING checkpoints (download from Google Drive `MyDrive/benchmark/`):')
    for p in missing_ckpts:
        print(f'  - {p}')
else:
    print('\nAll model checkpoints found OK.')

## 3. Configuration

In [ ]:
from src.eval.pipeline_runner import PipelineConfig

config = PipelineConfig(
    repo_path=str(REPO_PATH),
    encoder_path='outputs/encoders/hierarchical_temporal_v3/best_encoder.pt',
    ebm_path='outputs/ebm_models/ebm_v3/ebm_v3_final.pt',
    # Model architecture
    node_feature_dim=14,
    hidden_dim=128,
    num_spatial_layers=2,
    num_temporal_layers=4,
    num_heads=8,
    dropout=0.1,
    # EBM
    embed_dim=128,
    n_features=7,
    n_timesteps=24,
    # Langevin
    init_mode="bernoulli",
    init_p=0.10,
    prior_strength=0.0,
    noise_scale=1.2,
    langevin_steps=100,
    step_size=0.02,
    init_temp=1.5,
    final_temp=0.5,
    n_samples=10,  # keep max K in one run for best-of-K analysis
    # LP
    solver_name='appsi_highs',
    decoder_passthrough=True,  # preserve UC binaries; decoder only builds an LP warm-start
    device=DEVICE,
    seed=42,
)

# Family directories
FAMILIES = {
    'low': {
        'scenarios_dir': REPO_PATH / 'outputs/low_criticality_scenarios',
        'reports_dir': REPO_PATH / 'outputs/low_criticality_scenarios/reports',
        'graphs_dir': REPO_PATH / 'outputs/graphs/eval_low_criticality',
    },
    'medium': {
        'scenarios_dir': REPO_PATH / 'outputs/medium_criticality_scenarios',
        'reports_dir': REPO_PATH / 'outputs/medium_criticality_scenarios/reports',
        'graphs_dir': REPO_PATH / 'outputs/graphs/eval_medium_criticality',
    },
    'high': {
        'scenarios_dir': REPO_PATH / 'outputs/high_criticality_scenarios',
        'reports_dir': REPO_PATH / 'outputs/high_criticality_scenarios/reports',
        'graphs_dir': REPO_PATH / 'outputs/graphs/eval_high_criticality',
    },
}

OUTPUT_DIR = REPO_PATH / 'outputs/pipeline_eval_criticality'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

torch.manual_seed(config.seed)
np.random.seed(config.seed)

# Verify data availability
for name, paths in FAMILIES.items():
    sc_count = len(list(paths['scenarios_dir'].glob('scenario_*.json'))) if paths['scenarios_dir'].exists() else 0
    rp_count = len(list(paths['reports_dir'].glob('scenario_*.json'))) if paths['reports_dir'].exists() else 0
    print(f'  {name}: {sc_count} scenarios, {rp_count} reports')


## 4. Load Models

In [ ]:
from src.eval.pipeline_runner import PipelineRunner

runner = PipelineRunner(config)
runner.load_models()

## 4b. Sampler Diversity Smoke Test

Before launching the full evaluation, we run a **smoke test on 5 scenarios per family** with enough samples to support $K \in \{1,3,5,10\}$.

This is the quick go / no-go gate for the diversity configuration:
- mean pairwise Hamming distance should increase,
- EBM energy std should increase,
- best-of-$K$ cost gap should improve from $K=1$ to $K=10$,
- slack should remain controlled and LP stages should not collapse into pathological late repairs.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dataclasses import asdict, replace

from src.eval.metrics import compute_eval_metrics
from src.eval.advanced_analysis import compute_k_sampling_diagnostics

SMOKE_FAMILY_COLORS = globals().get('FAMILY_COLORS', {
    'low': '#2ecc71',
    'medium': '#f1c40f',
    'high': '#e74c3c',
})

SMOKE_TEST_MAX_SCENARIOS = 5
SMOKE_TEST_K_VALUES = [1, 3, 5, 10]
SMOKE_TEST_N_SAMPLES = max(int(config.n_samples), max(SMOKE_TEST_K_VALUES))

if SMOKE_TEST_N_SAMPLES != int(config.n_samples):
    print(f'[info] Building a temporary runner with n_samples={SMOKE_TEST_N_SAMPLES} for the smoke test.')
    smoke_config = replace(config, n_samples=SMOKE_TEST_N_SAMPLES)
    smoke_runner = PipelineRunner(smoke_config)
    smoke_runner.load_models()
else:
    smoke_config = config
    smoke_runner = runner

smoke_results = []
for name, paths in FAMILIES.items():
    smoke_family_results = smoke_runner.evaluate_family(
        scenarios_dir=paths['scenarios_dir'],
        reports_dir=paths['reports_dir'],
        graphs_dir=OUTPUT_DIR / 'graphs_smoke_test',
        family_name=name,
        max_scenarios=SMOKE_TEST_MAX_SCENARIOS,
    )
    smoke_results.extend(smoke_family_results)

smoke_runner.save_results(smoke_results, OUTPUT_DIR / 'pipeline_eval_smoke_test.pkl')
smoke_results_dict = [r if isinstance(r, dict) else asdict(r) for r in smoke_results]
smoke_eval = compute_eval_metrics(
    smoke_results_dict,
    {name: paths['reports_dir'] for name, paths in FAMILIES.items()},
)
smoke_df = smoke_eval['dataframe']
smoke_k_values = [k for k in SMOKE_TEST_K_VALUES if k <= SMOKE_TEST_N_SAMPLES]
smoke_k_summary, smoke_k_scenario = compute_k_sampling_diagnostics(
    smoke_results_dict,
    smoke_df,
    k_values=smoke_k_values,
)

display(smoke_k_summary.round(3))

if len(smoke_k_scenario):
    smoke_family_summary = (
        smoke_k_scenario.groupby(['family', 'k'])
        .agg(
            mean_hamming=('prefix_mean_hamming', 'mean'),
            mean_activation_rate=('prefix_activation_rate_mean', 'mean'),
            mean_energy_std=('prefix_energy_std', 'mean'),
            median_best_abs_cost_gap_pct=('best_of_k_abs_cost_gap_pct', 'median'),
            median_best_slack_mwh=('best_of_k_slack_mwh', 'median'),
            n_scenarios=('scenario_id', 'nunique'),
        )
        .reset_index()
    )
    display(smoke_family_summary.round(3))

    smoke_stage_table = (
        smoke_k_scenario.groupby(['k', 'best_of_k_stage_reached'])
        .size()
        .rename('count')
        .reset_index()
        .pivot(index='k', columns='best_of_k_stage_reached', values='count')
        .fillna(0)
    )
    display(smoke_stage_table.astype(int))

    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    axes = axes.ravel()

    axes[0].plot(smoke_k_summary['k'], smoke_k_summary['mean_hamming'], marker='o', lw=2, color='#1f77b4')
    axes[0].set_title('(A) Mean Pairwise Hamming', fontweight='bold')
    axes[0].set_xlabel('K')
    axes[0].set_ylabel('Distance')
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(smoke_k_summary['k'], smoke_k_summary['mean_energy_std'], marker='o', lw=2, color='#d62728')
    axes[1].set_title('(B) Final EBM Energy Std', fontweight='bold')
    axes[1].set_xlabel('K')
    axes[1].set_ylabel('Std(E)')
    axes[1].grid(True, alpha=0.3)

    axes[2].plot(smoke_k_summary['k'], smoke_k_summary['mean_activation_rate'], marker='o', lw=2, color='#2ca02c')
    axes[2].set_title('(C) Mean Activation Rate', fontweight='bold')
    axes[2].set_xlabel('K')
    axes[2].set_ylabel('Active fraction')
    axes[2].grid(True, alpha=0.3)

    axes[3].plot(smoke_k_summary['k'], smoke_k_summary['median_best_abs_cost_gap_pct'], marker='o', lw=2.5, color='#9467bd', label='Overall median |gap|')
    for fam in ['low', 'medium', 'high']:
        fam_df = smoke_family_summary[smoke_family_summary['family'] == fam]
        if len(fam_df) == 0:
            continue
        axes[3].plot(fam_df['k'], fam_df['median_best_abs_cost_gap_pct'], marker='o', lw=1.5, color=SMOKE_FAMILY_COLORS.get(fam), label=fam.capitalize())
    axes[3].set_title('(D) Best-of-K Absolute Cost Gap', fontweight='bold')
    axes[3].set_xlabel('K')
    axes[3].set_ylabel('|Cost gap| vs MILP (%)')
    axes[3].grid(True, alpha=0.3)
    axes[3].legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'fig_smoke_test_k_sampling.png', dpi=300, bbox_inches='tight')
    plt.show()

    base_k = min(smoke_k_values)
    top_k = max(smoke_k_values)
    pivot_abs_gap = smoke_k_scenario.pivot_table(
        index=['family', 'scenario_id'],
        columns='k',
        values='best_of_k_abs_cost_gap_pct',
        aggfunc='first',
    )
    if base_k in pivot_abs_gap.columns and top_k in pivot_abs_gap.columns:
        paired = pivot_abs_gap[[base_k, top_k]].dropna()
        improvement_rate = float((paired[top_k] <= paired[base_k]).mean() * 100) if len(paired) else np.nan
        print(f'Absolute cost gap improved from K={base_k} to K={top_k} on {improvement_rate:.1f}% of smoke-test scenarios.')
else:
    print('Smoke test produced no K-diagnostics. Check sampler outputs / LP failures before the full run.')


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path

from src.eval.metrics import compute_eval_metrics
from src.eval.advanced_analysis import compute_k_sampling_diagnostics
from src.eval.pipeline_runner import PipelineRunner

SMOKE_RESULTS_PATH = Path(r'C:\Users\coudr\OneDrive\projects\benchmark\benchmark_milp_gnn\outputs\pipeline_eval_criticality\pipeline_eval_smoke_test.pkl')
SMOKE_OUTPUT_DIR = SMOKE_RESULTS_PATH.parent
SMOKE_REPO_PATH = Path(r'C:\Users\coudr\OneDrive\projects\benchmark\benchmark_milp_gnn')
SMOKE_REPORTS_DIRS = {
    'low': SMOKE_REPO_PATH / 'outputs/low_criticality_scenarios/reports',
    'medium': SMOKE_REPO_PATH / 'outputs/medium_criticality_scenarios/reports',
    'high': SMOKE_REPO_PATH / 'outputs/high_criticality_scenarios/reports',
}
SMOKE_TARGET_K = [1, 3, 5, 10]
SMOKE_FAMILY_COLORS = {
    'low': '#2ecc71',
    'medium': '#f1c40f',
    'high': '#e74c3c',
}

if not SMOKE_RESULTS_PATH.exists():
    print(f'Smoke-test pickle not found: {SMOKE_RESULTS_PATH}')
else:
    smoke_loaded_results = PipelineRunner.load_results(SMOKE_RESULTS_PATH)
    print(f'Loaded {len(smoke_loaded_results)} smoke-test results from: {SMOKE_RESULTS_PATH}')

    if len(smoke_loaded_results) == 0:
        print('The smoke-test pickle is empty.')
    else:
        smoke_loaded_eval = compute_eval_metrics(smoke_loaded_results, SMOKE_REPORTS_DIRS)
        smoke_loaded_df = smoke_loaded_eval['dataframe']

        max_available_samples = max(
            (max(len(rec.get('all_objectives', [])), int(rec.get('n_samples', 0) or 0)) for rec in smoke_loaded_results),
            default=0,
        )
        smoke_loaded_k_values = [k for k in SMOKE_TARGET_K if k <= max_available_samples]
        if not smoke_loaded_k_values and max_available_samples:
            smoke_loaded_k_values = [max_available_samples]

        print(f'Max available samples per scenario in pickle: {max_available_samples}')
        print(f'K values evaluated: {smoke_loaded_k_values}')

        smoke_loaded_summary, smoke_loaded_scenario = compute_k_sampling_diagnostics(
            smoke_loaded_results,
            smoke_loaded_df,
            k_values=smoke_loaded_k_values or [1],
        )

        display(smoke_loaded_summary.round(3))

        if len(smoke_loaded_scenario):
            smoke_loaded_family_summary = (
                smoke_loaded_scenario.groupby(['family', 'k'])
                .agg(
                    mean_hamming=('prefix_mean_hamming', 'mean'),
                    mean_activation_rate=('prefix_activation_rate_mean', 'mean'),
                    mean_energy_std=('prefix_energy_std', 'mean'),
                    median_best_cost_gap_pct=('best_of_k_cost_gap_pct', 'median'),
                    median_best_abs_cost_gap_pct=('best_of_k_abs_cost_gap_pct', 'median'),
                    median_best_slack_mwh=('best_of_k_slack_mwh', 'median'),
                    n_scenarios=('scenario_id', 'nunique'),
                )
                .reset_index()
            )
            display(smoke_loaded_family_summary.round(3))

            smoke_loaded_stage_table = (
                smoke_loaded_scenario.groupby(['k', 'best_of_k_stage_reached'])
                .size()
                .rename('count')
                .reset_index()
                .pivot(index='k', columns='best_of_k_stage_reached', values='count')
                .fillna(0)
            )
            display(smoke_loaded_stage_table.astype(int))

            fig, axes = plt.subplots(2, 2, figsize=(15, 10))
            axes = axes.ravel()

            axes[0].plot(smoke_loaded_summary['k'], smoke_loaded_summary['mean_hamming'], marker='o', lw=2, color='#1f77b4')
            axes[0].set_title('(A) Mean Pairwise Hamming', fontweight='bold')
            axes[0].set_xlabel('K')
            axes[0].set_ylabel('Distance')
            axes[0].grid(True, alpha=0.3)

            axes[1].plot(smoke_loaded_summary['k'], smoke_loaded_summary['mean_energy_std'], marker='o', lw=2, color='#d62728')
            axes[1].set_title('(B) Final EBM Energy Std', fontweight='bold')
            axes[1].set_xlabel('K')
            axes[1].set_ylabel('Std(E)')
            axes[1].grid(True, alpha=0.3)

            axes[2].plot(smoke_loaded_summary['k'], smoke_loaded_summary['mean_activation_rate'], marker='o', lw=2, color='#2ca02c')
            axes[2].set_title('(C) Mean Activation Rate', fontweight='bold')
            axes[2].set_xlabel('K')
            axes[2].set_ylabel('Active fraction')
            axes[2].grid(True, alpha=0.3)

            axes[3].plot(smoke_loaded_summary['k'], smoke_loaded_summary['median_best_abs_cost_gap_pct'], marker='o', lw=2.5, color='#9467bd', label='Overall median |gap|')
            for fam in ['low', 'medium', 'high']:
                fam_df = smoke_loaded_family_summary[smoke_loaded_family_summary['family'] == fam]
                if len(fam_df) == 0:
                    continue
                axes[3].plot(fam_df['k'], fam_df['median_best_abs_cost_gap_pct'], marker='o', lw=1.5, color=SMOKE_FAMILY_COLORS.get(fam), label=fam.capitalize())
            axes[3].set_title('(D) Best-of-K Absolute Cost Gap', fontweight='bold')
            axes[3].set_xlabel('K')
            axes[3].set_ylabel('|Cost gap| vs MILP (%)')
            axes[3].grid(True, alpha=0.3)
            axes[3].legend(fontsize=8)

            plt.tight_layout()
            plt.savefig(SMOKE_OUTPUT_DIR / 'fig_smoke_test_k_sampling_loaded.png', dpi=300, bbox_inches='tight')
            plt.show()

            base_k = min(smoke_loaded_k_values)
            top_k = max(smoke_loaded_k_values)
            pivot_abs_gap = smoke_loaded_scenario.pivot_table(
                index=['family', 'scenario_id'],
                columns='k',
                values='best_of_k_abs_cost_gap_pct',
                aggfunc='first',
            )
            if base_k in pivot_abs_gap.columns and top_k in pivot_abs_gap.columns:
                paired = pivot_abs_gap[[base_k, top_k]].dropna()
                improvement_rate = float((paired[top_k] <= paired[base_k]).mean() * 100) if len(paired) else np.nan
                print(f'Absolute cost gap improved from K={base_k} to K={top_k} on {improvement_rate:.1f}% of loaded smoke-test scenarios.')
        else:
            print('The loaded smoke-test pickle does not contain usable K-diagnostics yet.')


## 5. Run Pipeline on All 3 Families

For each family (low/medium/high criticality), the pipeline runs:
1. Build Hierarchical Temporal Graphs
2. Generate HTE Embeddings
3. Extract Zone-Level Embeddings
4. EBM + Langevin Sampling -> binary candidates
5. Pass-through warm-start decoder -> never modify UC binaries
6. LP Worker Two-Stage -> solve continuous LP

In [ ]:
# Clear stale graph cache
import shutil
graphs_base = OUTPUT_DIR / 'graphs'


# Re-run — graphs_dir will now auto-create per-family subdirs
all_results = []
for name, paths in FAMILIES.items():
    results = runner.evaluate_family(
        scenarios_dir=paths['scenarios_dir'],
        reports_dir=paths['reports_dir'],
        graphs_dir=OUTPUT_DIR / 'graphs',   # auto-creates graphs/low/, graphs/medium/, graphs/high/
        family_name=name,
    )
    all_results.extend(results)

# Save all results
runner.save_results(all_results, OUTPUT_DIR / 'pipeline_eval_all_families.pkl')
print(f'\nTotal: {len(all_results)} scenarios evaluated')

## 6. Load MILP Reports & Build Comparison DataFrame

In [ ]:
from src.eval.metrics import (
    load_milp_reports, build_comparison_dataframe,
    compute_eval_metrics, compute_percentile_metrics,
    format_metrics_table,
)
from src.eval.pipeline_runner import PipelineRunner
from dataclasses import asdict

# Load pipeline results (prefer the saved pickle so the notebook can be reopened later)
results_path = OUTPUT_DIR / 'pipeline_eval_all_families.pkl'
if results_path.exists():
    pipeline_results = PipelineRunner.load_results(results_path)
    print(f'Loaded pipeline results from: {results_path}')
elif 'all_results' in globals():
    pipeline_results = [r if isinstance(r, dict) else asdict(r) for r in all_results]
    print('Using in-memory pipeline results from the current session.')
else:
    raise FileNotFoundError(
        f'No pipeline results found at {results_path}. Run Section 5 first.'
    )

# Load MILP data
milp_reports_dirs = {name: paths['reports_dir'] for name, paths in FAMILIES.items()}

# Compute metrics
eval_output = compute_eval_metrics(pipeline_results, milp_reports_dirs)
df = eval_output['dataframe']

print("\nTiming convention: `pipeline_solve_time` is the actual end-to-end wall-clock time")
print("of the selected best-of-K pipeline, including all candidate decoder+LP solves.")
print("It should not be divided by K. Oracle and parallel lower bounds are exposed as")
print("`pipeline_best_candidate_oracle_time` and `pipeline_parallel_ideal_time` for sensitivity analysis.\n")

timing_summary = (
    df.groupby('family')[[
        'pipeline_solve_time',
        'pipeline_first_candidate_time',
        'pipeline_best_candidate_oracle_time',
        'pipeline_parallel_ideal_time',
        'milp_solve_time',
        'speedup',
        'speedup_first_candidate',
        'speedup_best_candidate_oracle',
        'speedup_parallel_ideal',
    ]]
    .median()
    .rename(columns={
        'pipeline_solve_time': 'pipe_wall_clock_s',
        'pipeline_first_candidate_time': 'pipe_k1_s',
        'pipeline_best_candidate_oracle_time': 'pipe_best_oracle_s',
        'pipeline_parallel_ideal_time': 'pipe_parallel_ideal_s',
        'milp_solve_time': 'milp_s',
        'speedup': 'speedup_actual',
        'speedup_first_candidate': 'speedup_k1',
        'speedup_best_candidate_oracle': 'speedup_best_oracle',
        'speedup_parallel_ideal': 'speedup_parallel_ideal',
    })
)
display(timing_summary.round(2))

print(f'Comparison DataFrame: {df.shape}')
print(f'Families: {df["family"].value_counts().to_dict()}')
print(f'
Success rate: {df["success"].mean()*100:.1f}%')
df.head(10)


## 6b. Load RH-MO+LP Heuristic Baseline

Load the pre-computed results of the **Rolling-Horizon Merit-Order + LP** baseline
(`src.heuristics.runner`). The runner produced one JSON per family — upload them to
Google Drive at `outputs/rhmo_low.json`, `outputs/rhmo_medium.json`,
`outputs/rhmo_high.json` (any of these can be missing — the cell skips missing files).

The heuristic is merged into the existing comparison DataFrame with prefix
`heur_`, and compared against the MILP oracle the same way as the Pipeline:
- `heur_cost_gap_pct` = 100 · (heur_obj − milp_obj) / |milp_obj|
- `heur_speedup` = milp_solve_time / heur_total_time

In [ ]:
# --- Load heuristic results JSONs produced by src.heuristics.runner ---
HEURISTIC_FILES = {
    'low':    REPO_PATH / 'outputs/rhmo_low.json',
    'medium': REPO_PATH / 'outputs/rhmo_medium.json',
    'high':   REPO_PATH / 'outputs/rhmo_high.json',
}

heur_records = []
for fam, path in HEURISTIC_FILES.items():
    if not path.exists():
        print(f'  [skip] {fam}: {path} not found')
        continue
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    for rec in data:
        rec['family'] = fam  # force family tag (runner stores it but be safe)
        heur_records.append(rec)
    print(f'  [ok]   {fam}: loaded {len(data)} records from {path.name}')

heur_df = pd.DataFrame(heur_records)

if len(heur_df) == 0:
    print('\nNo heuristic results found - heuristic comparison will be skipped.')
    HAS_HEURISTIC = False
else:
    HAS_HEURISTIC = True
    # Prefix columns to avoid clashing with pipeline / MILP columns
    rename_map = {
        'lp_objective':             'heur_objective',
        'lp_status':                'heur_status',
        'lp_stage_used':            'heur_stage',
        'lp_stage_reached':         'heur_stage_reached',
        'lp_slack':                 'heur_slack',
        'lp_n_flips':               'heur_n_flips',
        # Per-stage slack progression (may be absent on older JSONs)
        'lp_slack_hard_fix':        'heur_slack_hard_fix',
        'lp_slack_repair_20':       'heur_slack_repair_20',
        'lp_slack_repair_100':      'heur_slack_repair_100',
        'lp_slack_full_soft':       'heur_slack_full_soft',
        'lp_slack_round_refix':     'heur_slack_round_refix',
        'lp_reached_hard_fix':      'heur_reached_hard_fix',
        'lp_reached_repair_20':     'heur_reached_repair_20',
        'lp_reached_repair_100':    'heur_reached_repair_100',
        'lp_reached_full_soft':     'heur_reached_full_soft',
        'lp_reached_round_refix':   'heur_reached_round_refix',
        'time_heuristic':           'heur_time_stageA',
        'time_decoder':             'heur_time_decoder',
        'time_lp_solve':            'heur_time_lp',
        'time_total':               'heur_solve_time',
        'success':                  'heur_success',
    }
    heur_df = heur_df.rename(columns=rename_map)

    keep = [
        'scenario_id', 'family',
        'heur_objective', 'heur_status', 'heur_stage', 'heur_stage_reached', 'heur_slack',
        'heur_n_flips',
        'heur_slack_hard_fix', 'heur_slack_repair_20',
        'heur_slack_repair_100', 'heur_slack_full_soft',
        'heur_slack_round_refix',
        'heur_reached_hard_fix', 'heur_reached_repair_20',
        'heur_reached_repair_100', 'heur_reached_full_soft',
        'heur_reached_round_refix',
        'heur_time_stageA', 'heur_time_decoder',
        'heur_time_lp', 'heur_solve_time', 'heur_success',
    ]
    heur_df = heur_df[[c for c in keep if c in heur_df.columns]]

    # Merge into main df on (scenario_id, family)
    df = df.merge(heur_df, on=['scenario_id', 'family'], how='left')

    # Derived: gap vs MILP and speedup vs MILP
    milp_abs = df['milp_objective'].abs().replace(0, np.nan)
    df['heur_cost_gap_pct'] = 100.0 * (df['heur_objective'] - df['milp_objective']) / milp_abs
    df['heur_cost_gap_abs'] = df['heur_objective'] - df['milp_objective']
    df['heur_speedup']      = df['milp_solve_time'] / df['heur_solve_time'].replace(0, np.nan)

    n_matched = df['heur_objective'].notna().sum()
    print(f'\nMerged heuristic rows: {n_matched} / {len(df)}')
    print(f'Heuristic success rate: {df["heur_success"].mean()*100:.1f}%')
    print(f'Mean heuristic cost gap vs MILP: {df["heur_cost_gap_pct"].mean():.2f}%')
    print(f'Mean heuristic speedup vs MILP:  {df["heur_speedup"].mean():.2f}x')

    reached_cols = [
        'heur_reached_hard_fix', 'heur_reached_repair_20', 'heur_reached_repair_100',
        'heur_reached_full_soft', 'heur_reached_round_refix',
    ]
    missing_reached = [c for c in reached_cols if c not in df.columns]
    if missing_reached:
        print(f'  [warn] missing stage-reached cols on heuristic JSONs: {missing_reached}')
        print('         re-run the heuristic runner to expose Stage 4/5 reach statistics.')


## 7. P90-P99 Metrics (Global & Per-Family)

In [ ]:
# --- Pipeline metrics (from compute_eval_metrics) ---
print(format_metrics_table(eval_output['global'], 'Global (all 300 scenarios) - PIPELINE'))
for fam_name, fam_metrics in eval_output['per_family'].items():
    print(format_metrics_table(fam_metrics, f'{fam_name} - PIPELINE'))

# --- Heuristic metrics (same percentiles, side-by-side) ---
def _print_heur_block(sub_df, title):
    metrics = {
        'cost_gap_pct': sub_df['heur_cost_gap_pct'],
        'speedup':      sub_df['heur_speedup'],
        'heur_time_s':  sub_df['heur_solve_time'],
        'slack_mwh':    sub_df['heur_slack'],
    }
    print('\n' + '=' * 70)
    print(f'  {title} - HEURISTIC (RH-MO+LP)')
    print('=' * 70)
    for name, s in metrics.items():
        s = pd.Series(s).dropna()
        if len(s) == 0:
            print(f'  {name}: (no data)')
            continue
        p10, p90 = np.percentile(s, [10, 90])
        p95, p99 = np.percentile(s, [95, 99])
        print(f'\n  {name}:')
        print(f'    Mean: {s.mean():.2f} (std: {s.std():.2f})')
        print(f'    Median: {s.median():.2f}')
        print(f'    [P10, P90]: [{p10:.2f}, {p90:.2f}]')
        print(f'    [P95, P99]: [{p95:.2f}, {p99:.2f}]')
    if 'heur_stage' in sub_df.columns and sub_df['heur_stage'].notna().any():
        stage_counts = sub_df['heur_stage'].value_counts(normalize=True) * 100
        print('\n  Stage Distribution:')
        for stage, pct in stage_counts.items():
            print(f'    {stage}: {pct:.1f}%')
    print('=' * 70)

if HAS_HEURISTIC:
    _print_heur_block(df, 'Global (all scenarios)')
    for fam in ['low', 'medium', 'high']:
        sub = df[df['family'] == fam]
        if len(sub) > 0 and sub['heur_objective'].notna().any():
            _print_heur_block(sub, fam)
else:
    print('\n[Heuristic results not loaded - heuristic metrics skipped]')

## 8. Solve Time Relationship: MILP vs Heuristic vs Pipeline

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import seaborn as sns
from scipy import stats
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 150
FAMILY_COLORS = {'low': '#2ecc71', 'medium': '#f1c40f', 'high': '#e74c3c'}
# LP-worker stages (cascade order: hard_fix -> repair_20 -> repair_100 -> full_soft -> round_refix).
STAGE_COLORS = {
    'hard_fix':    '#2ecc71',
    'repair_20':   '#f1c40f',
    'repair_100':  '#e67e22',
    'full_soft':   '#e74c3c',
    'round_refix': '#9b59b6',
}

# Robust check: only plot heuristic row if its columns actually exist on df
SHOW_HEUR = bool(globals().get('HAS_HEURISTIC', False)) and \
    {'heur_solve_time', 'heur_cost_gap_pct', 'heur_speedup'}.issubset(df.columns)
if globals().get('HAS_HEURISTIC', False) and not SHOW_HEUR:
    print('[warn] HAS_HEURISTIC=True but heuristic columns missing on df '
          '(re-run the heuristic merge cell). Plotting Pipeline only.')

# ============================================================================
# Two rows: row 0 = Pipeline vs MILP, row 1 = Heuristic vs MILP.
# Each row has 3 panels: (A) actual wall-clock time, (B) cost gap, (C) actual speedup.
# ============================================================================
n_rows = 2 if SHOW_HEUR else 1
fig, axes = plt.subplots(n_rows, 3, figsize=(18, 5 * n_rows), squeeze=False)

fam_handles = [Line2D([0], [0], marker='o', color='w', markerfacecolor=c,
                      markersize=8, label=f) for f, c in FAMILY_COLORS.items()]


def _plot_row(row_axes, time_col, gap_col, speed_col, label):
    # --- Panel A: wall-clock solve time vs MILP ---
    ax = row_axes[0]
    for fam, color in FAMILY_COLORS.items():
        mask = df['family'] == fam
        ax.scatter(df.loc[mask, 'milp_solve_time'], df.loc[mask, time_col],
                   c=color, marker='o', s=40, alpha=0.7,
                   edgecolors='white', linewidth=0.5)
    mx = df['milp_solve_time'].max()
    ax.plot([1e-2, mx], [1e-2, mx], 'k--', alpha=0.3, label='x=y')
    ax.set_xlabel('MILP Solve Time (s)')
    ax.set_ylabel(f'{label} Wall-Clock Time (s)')
    ax.set_title(f'(A) {label} vs MILP — Solve Time')
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.legend(handles=fam_handles + [Line2D([0], [0], color='k', linestyle='--',
                                             alpha=0.4, label='x=y')],
              fontsize=8, loc='upper left')

    # --- Panel B: cost gap vs criticality ---
    ax = row_axes[1]
    for fam, color in FAMILY_COLORS.items():
        mask = df['family'] == fam
        ax.scatter(df.loc[mask, 'criticality_index'], df.loc[mask, gap_col],
                   c=color, marker='o', s=40, alpha=0.7,
                   edgecolors='white', linewidth=0.5)
    ax.axhline(y=0, color='green', linestyle='--', alpha=0.5)
    ax.axhline(y=5, color='orange', linestyle=':', alpha=0.5, label='5% tolerance')
    ax.set_xlabel('Criticality Index')
    ax.set_ylabel(f'{label} Cost Gap vs MILP (%)')
    ax.set_title(f'(B) {label} — Cost Gap vs Criticality')
    ax.legend(handles=fam_handles + [Line2D([0], [0], color='orange',
                                             linestyle=':', label='5% tol')],
              fontsize=8, loc='upper left')

    # --- Panel C: speedup vs criticality ---
    ax = row_axes[2]
    for fam, color in FAMILY_COLORS.items():
        mask = df['family'] == fam
        ax.scatter(df.loc[mask, 'criticality_index'], df.loc[mask, speed_col],
                   c=color, marker='o', s=40, alpha=0.7,
                   edgecolors='white', linewidth=0.5)
    valid = df.dropna(subset=['criticality_index', speed_col])
    if len(valid) > 3:
        slope, intercept, r_val, _, _ = stats.linregress(
            valid['criticality_index'], valid[speed_col])
        x_line = np.linspace(valid['criticality_index'].min(),
                             valid['criticality_index'].max(), 50)
        ax.plot(x_line, slope * x_line + intercept, 'k--', lw=2, alpha=0.6,
                label=f'R²={r_val**2:.2f}')
    ax.axhline(y=1, color='red', linestyle=':', alpha=0.5, label='Break-even')
    ax.set_xlabel('Criticality Index')
    ax.set_ylabel(f'{label} Actual Speedup (MILP / {label})')
    ax.set_title(f'(C) {label} — Speedup vs Criticality')
    ax.set_yscale('log')
    ax.legend(fontsize=8, loc='upper left')


# Row 0 — Pipeline
_plot_row(axes[0], 'pipeline_solve_time', 'cost_gap_pct', 'speedup', 'Pipeline')
# Row 1 — Heuristic (if available)
if SHOW_HEUR:
    _plot_row(axes[1], 'heur_solve_time', 'heur_cost_gap_pct', 'heur_speedup', 'Heuristic')

# Optional bold row labels
for i, label in enumerate(['Pipeline'] + (['Heuristic'] if SHOW_HEUR else [])):
    axes[i, 0].text(-0.18, 0.5, label, transform=axes[i, 0].transAxes,
                    fontsize=14, fontweight='bold', rotation=90,
                    ha='center', va='center')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig1_solve_time_cost_gap_speedup.png', dpi=300, bbox_inches='tight')
plt.show()

## 9. Stage Distribution by Family

In [ ]:
# Robust check: heuristic stage columns must actually exist on df.
SHOW_HEUR = bool(globals().get('HAS_HEURISTIC', False)) and \
    {'heur_speedup'}.issubset(df.columns)

stages = ['hard_fix', 'repair_20', 'repair_100', 'full_soft', 'round_refix']
families_list = ['low', 'medium', 'high']
n_fam = len(families_list)
x = np.arange(n_fam)

PIPE_STAGE_COL = 'pipeline_stage_reached' if 'pipeline_stage_reached' in df.columns else 'pipeline_stage'
HEUR_STAGE_COL = 'heur_stage_reached' if 'heur_stage_reached' in df.columns else 'heur_stage'

print(f'Pipeline stage source used for plots: {PIPE_STAGE_COL}')
if SHOW_HEUR:
    print(f'Heuristic stage source used for plots: {HEUR_STAGE_COL}')


def _stage_pcts(sub_df, stage_col):
    pcts = []
    for fam in families_list:
        fam_df = sub_df[sub_df['family'] == fam]
        if len(fam_df) == 0 or stage_col not in fam_df.columns:
            pcts.append({s: 0.0 for s in stages})
            continue
        counts = fam_df[stage_col].fillna('').value_counts()
        pcts.append({s: counts.get(s, 0) / len(fam_df) * 100 for s in stages})
    return pcts


def _plot_stage_bar(ax, pcts, title):
    bottom = np.zeros(n_fam)
    for stage in stages:
        vals = np.array([p[stage] for p in pcts])
        ax.bar(x, vals, 0.6, bottom=bottom, label=stage,
               color=STAGE_COLORS.get(stage, 'gray'), alpha=0.85,
               edgecolor='white', linewidth=0.5)
        bottom += vals
    ax.set_xticks(x)
    ax.set_xticklabels(['Low Crit.', 'Medium Crit.', 'High Crit.'])
    ax.set_ylabel('Proportion (%)')
    ax.set_title(title)
    ax.legend(loc='upper right', fontsize=8, ncol=2)
    ax.set_ylim(0, 105)


def _plot_speedup_box(ax, speed_col, title):
    data_boxes = [df[df['family'] == fam][speed_col].dropna() for fam in families_list]
    bp = ax.boxplot(data_boxes, tick_labels=['Low', 'Medium', 'High'],
                    patch_artist=True, showmeans=True)
    for patch, fam in zip(bp['boxes'], families_list):
        patch.set_facecolor(FAMILY_COLORS[fam])
        patch.set_alpha(0.6)
    ax.set_ylabel('Speedup (x)')
    ax.set_title(title)
    ax.set_yscale('log')
    ax.axhline(y=1, color='red', linestyle=':', alpha=0.5)


# ============================================================================
# Two rows: row 0 = Pipeline, row 1 = Heuristic (if available).
# Each row: (A) deepest LP stage reached by family, (B) speedup boxplot by family.
# ============================================================================
n_rows = 2 if SHOW_HEUR else 1
fig, axes = plt.subplots(n_rows, 2, figsize=(14, 5 * n_rows), squeeze=False)

pipe_pcts = _stage_pcts(df, PIPE_STAGE_COL)
_plot_stage_bar(axes[0, 0], pipe_pcts, '(A) Pipeline ? Deepest LP Stage Reached by Family')
_plot_speedup_box(axes[0, 1], 'speedup', '(B) Pipeline ? Speedup Distribution by Family')

if SHOW_HEUR:
    heur_pcts = _stage_pcts(df, HEUR_STAGE_COL)
    _plot_stage_bar(axes[1, 0], heur_pcts, '(A) Heuristic ? Deepest LP Stage Reached by Family')
    _plot_speedup_box(axes[1, 1], 'heur_speedup', '(B) Heuristic ? Speedup Distribution by Family')

# Bold row labels
for i, label in enumerate(['Pipeline'] + (['Heuristic'] if SHOW_HEUR else [])):
    axes[i, 0].text(-0.18, 0.5, label, transform=axes[i, 0].transAxes,
                    fontsize=14, fontweight='bold', rotation=90,
                    ha='center', va='center')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig2_stage_distribution_speedup.png', dpi=300, bbox_inches='tight')
plt.show()

# --- Print stage distribution table for both methods ---
print('\nDeepest stage reached (% of scenarios):')
print('-' * 78)
header = f"{'Stage':<14}" + ''.join([f"{f'{fam} pipe':>11}" for fam in families_list])
if SHOW_HEUR:
    header += ''.join([f"{f'{fam} heur':>11}" for fam in families_list])
print(header)
print('-' * 78)
for stage in stages:
    row = f"{stage:<14}"
    for i, fam in enumerate(families_list):
        row += f"{pipe_pcts[i][stage]:>10.1f}%"
    if SHOW_HEUR:
        for i, fam in enumerate(families_list):
            row += f"{heur_pcts[i][stage]:>10.1f}%"
    print(row)


## 9b. Slack Diagnostics by Stage

With **Stage 5 (round_refix)** now active, the LP worker always returns an integer-feasible solution — but at the cost of potentially large unserved energy / overgen slack when the rounded binaries don't admit a tight dispatch. Slack therefore replaces "stage reached" as the primary feasibility signal.

For each stage and family we report:
- `n` scenarios reaching that stage
- mean / median / P90 / P95 / max slack (MWh)
- fraction with slack > 1 MWh (i.e. truly using unserved/overgen)

Both Pipeline and Heuristic are shown side-by-side.

In [ ]:
# ============================================================
# Slack diagnostics by stage (Pipeline + Heuristic)
#
# We count a stage as "executed" using the explicit reached flags when
# available. This fixes the previous ambiguity where Stage 4 / Stage 5 could
# disappear from tables because their slack happened to be 0.0 or because the
# final stage label was `failed`.
# ============================================================
stages = ['hard_fix', 'repair_20', 'repair_100', 'full_soft', 'round_refix']
families_list = ['low', 'medium', 'high']
SLACK_THRESH = 1.0  # MWh ? above this we consider slack "real"
stage_rank = {stage: idx for idx, stage in enumerate(stages, start=1)}

PIPE_SLACK_COLS = {s: f'pipeline_slack_{s}' for s in stages}
HEUR_SLACK_COLS = {s: f'heur_slack_{s}' for s in stages}
PIPE_REACHED_COLS = {s: f'pipeline_reached_{s}' for s in stages}
HEUR_REACHED_COLS = {s: f'heur_reached_{s}' for s in stages}
PIPE_STAGE_COL = 'pipeline_stage_reached' if 'pipeline_stage_reached' in df.columns else 'pipeline_stage'
HEUR_STAGE_COL = 'heur_stage_reached' if 'heur_stage_reached' in df.columns else 'heur_stage'


def _executed_mask(sub_df, stage, slack_cols, reached_cols, deepest_stage_col):
    reached_col = reached_cols[stage]
    if reached_col in sub_df.columns:
        return sub_df[reached_col].fillna(False).astype(bool)
    if deepest_stage_col in sub_df.columns:
        deepest_rank = sub_df[deepest_stage_col].map(lambda s: stage_rank.get(str(s), 0)).fillna(0)
        return deepest_rank >= stage_rank[stage]
    slack_col = slack_cols[stage]
    if slack_col in sub_df.columns:
        return sub_df[slack_col].fillna(0.0) != 0.0
    return pd.Series(False, index=sub_df.index)


def _slack_stats_per_stage(sub_df, slack_cols, reached_cols, deepest_stage_col):
    rows = []
    for stage in stages:
        slack_col = slack_cols[stage]
        if slack_col not in sub_df.columns:
            rows.append({
                'stage': stage,
                'n_executed': 0,
                'pct_executed': 0.0,
                'mean_slack': np.nan,
                'median_slack': np.nan,
                'p90_slack': np.nan,
                'p95_slack': np.nan,
                'max_slack': np.nan,
                'pct_gt_threshold': np.nan,
            })
            continue

        mask = _executed_mask(sub_df, stage, slack_cols, reached_cols, deepest_stage_col)
        s = sub_df.loc[mask, slack_col].dropna()
        n_exec = int(mask.sum())
        if n_exec == 0 or len(s) == 0:
            rows.append({
                'stage': stage,
                'n_executed': n_exec,
                'pct_executed': 100.0 * n_exec / max(len(sub_df), 1),
                'mean_slack': np.nan,
                'median_slack': np.nan,
                'p90_slack': np.nan,
                'p95_slack': np.nan,
                'max_slack': np.nan,
                'pct_gt_threshold': np.nan,
            })
            continue

        rows.append({
            'stage': stage,
            'n_executed': n_exec,
            'pct_executed': 100.0 * n_exec / max(len(sub_df), 1),
            'mean_slack': float(s.mean()),
            'median_slack': float(s.median()),
            'p90_slack': float(np.percentile(s, 90)),
            'p95_slack': float(np.percentile(s, 95)),
            'max_slack': float(s.max()),
            'pct_gt_threshold': float((s > SLACK_THRESH).mean() * 100),
        })
    return pd.DataFrame(rows)


pipe_stage_slack = _slack_stats_per_stage(df, PIPE_SLACK_COLS, PIPE_REACHED_COLS, PIPE_STAGE_COL)
print('Pipeline ? slack diagnostics by executed stage')
display(pipe_stage_slack)

for fam in families_list:
    print(f'Pipeline ? {fam}')
    display(_slack_stats_per_stage(
        df[df['family'] == fam],
        PIPE_SLACK_COLS,
        PIPE_REACHED_COLS,
        PIPE_STAGE_COL,
    ))

HAS_HEUR_SLACK = bool(globals().get('HAS_HEURISTIC', False)) and 'heur_slack_hard_fix' in df.columns
if HAS_HEUR_SLACK:
    heur_stage_slack = _slack_stats_per_stage(df, HEUR_SLACK_COLS, HEUR_REACHED_COLS, HEUR_STAGE_COL)
    print('Heuristic ? slack diagnostics by executed stage')
    display(heur_stage_slack)
    for fam in families_list:
        print(f'Heuristic ? {fam}')
        display(_slack_stats_per_stage(
            df[df['family'] == fam],
            HEUR_SLACK_COLS,
            HEUR_REACHED_COLS,
            HEUR_STAGE_COL,
        ))
else:
    heur_stage_slack = pd.DataFrame()
    print('[info] Heuristic per-stage execution flags not available in the current JSONs.')

# --- Compact figure: execution rate and P95 slack by stage -------------------
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

plot_df = pipe_stage_slack.copy()
plot_df['stage_label'] = plot_df['stage'].str.replace('_', ' ')
axes[0].bar(plot_df['stage_label'], plot_df['pct_executed'],
            color=[STAGE_COLORS.get(s, 'gray') for s in plot_df['stage']])
axes[0].set_ylabel('Executed on scenarios (%)', fontweight='bold')
axes[0].set_title('(A) Pipeline ? Stage Execution Rate', fontweight='bold')
axes[0].tick_params(axis='x', rotation=25)
axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(plot_df['stage_label'], plot_df['p95_slack'].fillna(0.0),
            color=[STAGE_COLORS.get(s, 'gray') for s in plot_df['stage']])
axes[1].set_ylabel('P95 slack (MWh)', fontweight='bold')
axes[1].set_title('(B) Pipeline ? P95 Slack by Executed Stage', fontweight='bold')
axes[1].tick_params(axis='x', rotation=25)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig2b_slack_by_stage.png', dpi=300, bbox_inches='tight')
plt.show()

# --- Stage 4 vs Stage 5 paired comparison -----------------------------------
print('Stage 4 vs Stage 5 paired comparison')
pipe_mask = _executed_mask(df, 'full_soft', PIPE_SLACK_COLS, PIPE_REACHED_COLS, PIPE_STAGE_COL)
pipe_sub = df.loc[pipe_mask, ['family', 'pipeline_slack_full_soft', 'pipeline_slack_round_refix']]
if len(pipe_sub):
    print(f'  Pipeline ? {len(pipe_sub)} scenarios reached Stage 4 and therefore attempted Stage 5:')
    print(f'    Stage 4 mean slack: {pipe_sub["pipeline_slack_full_soft"].mean():.2f} MWh')
    print(f'    Stage 5 mean slack: {pipe_sub["pipeline_slack_round_refix"].mean():.2f} MWh')
    delta = pipe_sub['pipeline_slack_round_refix'] - pipe_sub['pipeline_slack_full_soft']
    print(f'    Delta (Stage 5 - Stage 4): mean={delta.mean():+.2f} MWh, median={delta.median():+.2f} MWh')
else:
    print('  Pipeline ? no Stage 4 / Stage 5 executions recorded in the current results.')


## 9c. Feasibility Overview (Pipeline vs Heuristic)

With Stage 5 round_refix now the cascade terminus, a scenario is labelled as:
- **feasible** — Stage 5 (or any earlier stage) returned a solution with slack ≤ threshold.
- **feasible (degraded)** — Stage 5 returned a solution but slack > threshold (round_refix could not dispatch the rounded binaries without invoking unserved/overgen).
- **infeasible** — Stage 5 failed entirely (or `status == 'infeasible'` / `stage_used == 'failed'` / objective is NaN/inf).

The figure below shows the per-family counts for both Pipeline and Heuristic in a 2-row layout.

In [ ]:
# ============================================================
# Feasibility overview: Pipeline vs Heuristic, per family.
# Categories: feasible / feasible_degraded / infeasible.
# ============================================================
SLACK_THRESH_FEAS = 1.0  # MWh
families_list = ['low', 'medium', 'high']

SHOW_HEUR = bool(globals().get('HAS_HEURISTIC', False)) and \
    {'heur_status', 'heur_stage', 'heur_slack'}.issubset(df.columns)

CAT_COLORS = {
    'feasible':          '#2ecc71',
    'feasible_degraded': '#f39c12',
    'infeasible':        '#c0392b',
}
CAT_ORDER = ['feasible', 'feasible_degraded', 'infeasible']


def _classify(status_col, stage_col, slack_col, obj_col):
    """Return a categorical Series over df rows."""
    status = df[status_col].fillna('').astype(str)
    stage  = df[stage_col].fillna('').astype(str)
    slack  = pd.to_numeric(df[slack_col], errors='coerce')
    obj    = pd.to_numeric(df[obj_col], errors='coerce')

    infeasible = (
        status.eq('infeasible') |
        status.eq('error') |
        status.eq('') |
        stage.eq('failed') |
        obj.isna() |
        np.isinf(obj)
    )
    degraded = (~infeasible) & (slack.fillna(0) > SLACK_THRESH_FEAS)
    feasible = ~(infeasible | degraded)

    cat = pd.Series('feasible', index=df.index, dtype=object)
    cat[degraded]   = 'feasible_degraded'
    cat[infeasible] = 'infeasible'
    return cat


def _counts_by_family(cat_series):
    """Return dict {family: {category: count}} including zeros."""
    out = {}
    for fam in families_list:
        sub = cat_series[df['family'] == fam]
        vc = sub.value_counts().to_dict()
        out[fam] = {c: int(vc.get(c, 0)) for c in CAT_ORDER}
    return out


pipe_cat   = _classify('pipeline_status', 'pipeline_stage',
                       'pipeline_slack', 'pipeline_objective')
pipe_counts = _counts_by_family(pipe_cat)

if SHOW_HEUR:
    heur_cat    = _classify('heur_status', 'heur_stage',
                            'heur_slack', 'heur_objective')
    heur_counts = _counts_by_family(heur_cat)

# ------------------------------------------------------------
# Figure: 2 rows (Pipeline / Heuristic) x 2 cols (counts / %)
# ------------------------------------------------------------
n_rows = 2 if SHOW_HEUR else 1
fig, axes = plt.subplots(n_rows, 2, figsize=(14, 4.5 * n_rows), squeeze=False)
x = np.arange(len(families_list))
width = 0.6


def _plot_counts(ax, counts, title):
    bottom = np.zeros(len(families_list))
    for cat in CAT_ORDER:
        vals = np.array([counts[fam][cat] for fam in families_list])
        ax.bar(x, vals, width, bottom=bottom, label=cat,
               color=CAT_COLORS[cat], edgecolor='white', linewidth=0.6)
        for xi, v, b in zip(x, vals, bottom):
            if v > 0:
                ax.text(xi, b + v / 2, str(v), ha='center', va='center',
                        color='white', fontsize=9, fontweight='bold')
        bottom += vals
    ax.set_xticks(x); ax.set_xticklabels(['Low', 'Medium', 'High'])
    ax.set_ylabel('# scenarios')
    ax.set_title(title)
    ax.legend(fontsize=8, loc='upper left')


def _plot_pct(ax, counts, title):
    bottom = np.zeros(len(families_list))
    totals = np.array([sum(counts[fam].values()) for fam in families_list],
                      dtype=float)
    totals[totals == 0] = 1.0
    for cat in CAT_ORDER:
        vals = np.array([counts[fam][cat] for fam in families_list]) / totals * 100
        ax.bar(x, vals, width, bottom=bottom, label=cat,
               color=CAT_COLORS[cat], edgecolor='white', linewidth=0.6)
        for xi, v, b in zip(x, vals, bottom):
            if v >= 4:  # only annotate non-tiny slices
                ax.text(xi, b + v / 2, f'{v:.0f}%', ha='center', va='center',
                        color='white', fontsize=9, fontweight='bold')
        bottom += vals
    ax.set_xticks(x); ax.set_xticklabels(['Low', 'Medium', 'High'])
    ax.set_ylabel('% of scenarios')
    ax.set_ylim(0, 105)
    ax.set_title(title)
    ax.legend(fontsize=8, loc='lower right')


_plot_counts(axes[0, 0], pipe_counts,
             '(A) Pipeline — scenarios by feasibility (count)')
_plot_pct(axes[0, 1], pipe_counts,
          '(B) Pipeline — scenarios by feasibility (%)')

if SHOW_HEUR:
    _plot_counts(axes[1, 0], heur_counts,
                 '(A) Heuristic — scenarios by feasibility (count)')
    _plot_pct(axes[1, 1], heur_counts,
              '(B) Heuristic — scenarios by feasibility (%)')

for i, label in enumerate(['Pipeline'] + (['Heuristic'] if SHOW_HEUR else [])):
    axes[i, 0].text(-0.15, 0.5, label, transform=axes[i, 0].transAxes,
                    fontsize=14, fontweight='bold', rotation=90,
                    ha='center', va='center')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig2c_feasibility.png', dpi=300, bbox_inches='tight')
plt.show()

# --- Summary table ---
print('\nFeasibility counts (threshold for "degraded" = '
      f'{SLACK_THRESH_FEAS} MWh slack):')
print('-' * 78)
header = f"{'method':<12}{'family':<10}"
for c in CAT_ORDER:
    header += f"{c:>20}"
header += f"{'total':>8}"
print(header)
print('-' * 78)


def _print_rows(name, counts):
    for fam in families_list:
        total = sum(counts[fam].values())
        row = f"{name:<12}{fam:<10}"
        for c in CAT_ORDER:
            pct = 100 * counts[fam][c] / total if total else 0.0
            row += f"{counts[fam][c]:>10} ({pct:>5.1f}%)"
        row += f"{total:>8}"
        print(row)


_print_rows('Pipeline', pipe_counts)
if SHOW_HEUR:
    _print_rows('Heuristic', heur_counts)

## 10. Pareto Frontier: Quality vs Speed

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

for fam, color in FAMILY_COLORS.items():
    mask = df['family'] == fam
    ax.scatter(df.loc[mask, 'pipeline_solve_time'], df.loc[mask, 'cost_gap_pct'],
              c=color, label=f'Pipeline ({fam})', s=80, alpha=0.7,
              edgecolors='white', linewidth=1)

# MILP reference (all at their solve time, gap=0)
ax.scatter(df['milp_solve_time'], [0]*len(df), c='gray', label='MILP (ref)',
          s=30, alpha=0.3, marker='^')

ax.axhline(y=0, color='green', linestyle='--', lw=1.5, alpha=0.6)
ax.axhline(y=5, color='orange', linestyle=':', lw=1, alpha=0.6, label='5% tolerance')
ax.axhline(y=-5, color='orange', linestyle=':', lw=1, alpha=0.6)

ax.set_xlabel('Solve Time (seconds)', fontsize=13, fontweight='bold')
ax.set_ylabel('Cost Gap vs MILP (%)', fontsize=13, fontweight='bold')
ax.set_title('Pareto Frontier: Solution Quality vs Speed', fontsize=15, fontweight='bold')
ax.set_xscale('log')
ax.legend(loc='upper right', fontsize=9)
ax.grid(True, alpha=0.3)

# Highlight Pareto-optimal region
ax.axhspan(-5, 5, xmin=0, xmax=0.3, alpha=0.08, color='green')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig3_pareto_quality_speed.png', dpi=300, bbox_inches='tight')
plt.show()

# Count Pareto-optimal
pareto = (df['cost_gap_pct'].abs() < 5) & (df['pipeline_solve_time'] < 120)
print(f'Pareto-optimal (|gap|<5%, time<120s): {pareto.sum()}/{len(df)}')

## 10b. Solution Diversity and Best-of-K

The EBM is evaluated as a **candidate sampler**, not just as a point predictor.

We track three complementary views:
- intra-scenario cost spread across the candidate pool,
- mean pairwise binary Hamming diversity over sampled $u$ tensors,
- best-of-$K$ quality / heuristic-win-rate / actual wall-clock curves for $K \in \{1,2,3,5,10\}$ whenever the saved results contain enough candidates.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.eval.advanced_analysis import (
    build_candidate_pool_frame,
    compute_solution_diversity_frame,
    compute_k_sampling_diagnostics,
    compute_best_of_k_curve,
)

MAX_AVAILABLE_SAMPLES = max(
    (len(rec.get('all_objectives', [])) for rec in pipeline_results),
    default=0,
)
BEST_OF_K_VALUES = [k for k in [1, 2, 3, 5, 10] if k <= MAX_AVAILABLE_SAMPLES]
if not BEST_OF_K_VALUES and MAX_AVAILABLE_SAMPLES:
    BEST_OF_K_VALUES = [MAX_AVAILABLE_SAMPLES]

candidate_pool_df = build_candidate_pool_frame(pipeline_results, df)
diversity_df = compute_solution_diversity_frame(pipeline_results, df)
bestofk_summary, bestofk_scenario = compute_best_of_k_curve(
    pipeline_results,
    df,
    k_values=BEST_OF_K_VALUES or [1],
)

print(f'Max available candidate count per scenario: {MAX_AVAILABLE_SAMPLES}')
if MAX_AVAILABLE_SAMPLES < 10:
    print('Current results do not yet support the full K={1,2,3,5,10} analysis.')
    print('Re-run Section 5 after setting n_samples=10 to unlock the full multi-candidate study.')

display(diversity_df.head())
display(bestofk_summary)


In [ ]:
from scipy.stats import spearmanr

# Candidate-level diagnostics saved per sample:
#   energy, lp_objective, slack_mwh, stage, cost_gap_pct
candidate_df = candidate_pool_df.copy()

for col in ['energy', 'lp_objective', 'cost_gap_pct', 'slack_mwh']:
    candidate_df[col] = pd.to_numeric(candidate_df[col], errors='coerce')

valid_cost = candidate_df[['energy', 'lp_objective']].dropna()
valid_gap = candidate_df[['energy', 'cost_gap_pct']].dropna()
valid_slack = candidate_df[['energy', 'slack_mwh']].dropna()

spearman_energy_cost = spearmanr(valid_cost['energy'], valid_cost['lp_objective'], nan_policy='omit') if len(valid_cost) >= 2 else None
spearman_energy_gap = spearmanr(valid_gap['energy'], valid_gap['cost_gap_pct'], nan_policy='omit') if len(valid_gap) >= 2 else None
spearman_energy_slack = spearmanr(valid_slack['energy'], valid_slack['slack_mwh'], nan_policy='omit') if len(valid_slack) >= 2 else None

print('Global candidate-level Spearman correlations')
if spearman_energy_cost is not None:
    print(f'  energy vs lp_objective: rho={spearman_energy_cost.statistic:.3f}, p={spearman_energy_cost.pvalue:.2e}, n={len(valid_cost)}')
else:
    print('  energy vs lp_objective: insufficient data')
if spearman_energy_gap is not None:
    print(f'  energy vs cost_gap_pct: rho={spearman_energy_gap.statistic:.3f}, p={spearman_energy_gap.pvalue:.2e}, n={len(valid_gap)}')
else:
    print('  energy vs cost_gap_pct: insufficient data')
if spearman_energy_slack is not None:
    print(f'  energy vs slack_mwh: rho={spearman_energy_slack.statistic:.3f}, p={spearman_energy_slack.pvalue:.2e}, n={len(valid_slack)}')
else:
    print('  energy vs slack_mwh: insufficient data')

family_corr_rows = []
for fam in ['low', 'medium', 'high']:
    fam_df = candidate_df[candidate_df['family'] == fam].copy()
    row = {'family': fam, 'n_candidates': int(len(fam_df))}
    fam_cost = fam_df[['energy', 'lp_objective']].dropna()
    fam_gap = fam_df[['energy', 'cost_gap_pct']].dropna()
    fam_slack = fam_df[['energy', 'slack_mwh']].dropna()
    corr_cost = spearmanr(fam_cost['energy'], fam_cost['lp_objective'], nan_policy='omit') if len(fam_cost) >= 2 else None
    corr_gap = spearmanr(fam_gap['energy'], fam_gap['cost_gap_pct'], nan_policy='omit') if len(fam_gap) >= 2 else None
    corr_slack = spearmanr(fam_slack['energy'], fam_slack['slack_mwh'], nan_policy='omit') if len(fam_slack) >= 2 else None
    row['rho_energy_lp_objective'] = float(corr_cost.statistic) if corr_cost is not None else np.nan
    row['p_energy_lp_objective'] = float(corr_cost.pvalue) if corr_cost is not None else np.nan
    row['rho_energy_cost_gap_pct'] = float(corr_gap.statistic) if corr_gap is not None else np.nan
    row['p_energy_cost_gap_pct'] = float(corr_gap.pvalue) if corr_gap is not None else np.nan
    row['rho_energy_slack_mwh'] = float(corr_slack.statistic) if corr_slack is not None else np.nan
    row['p_energy_slack_mwh'] = float(corr_slack.pvalue) if corr_slack is not None else np.nan
    family_corr_rows.append(row)

candidate_energy_correlation_df = pd.DataFrame(family_corr_rows)
display(candidate_df[['family', 'scenario_id', 'sample_idx', 'energy', 'lp_objective', 'cost_gap_pct', 'slack_mwh', 'stage', 'stage_reached']].head(15))
display(candidate_energy_correlation_df.round(4))


In [ ]:
# --- Supporting diversity figure --------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
spread_plot_df = diversity_df.dropna(subset=['pool_cost_spread_vs_milp_pct']).copy()
if len(spread_plot_df):
    sns.boxplot(data=spread_plot_df, x='family', y='pool_cost_spread_vs_milp_pct',
                order=['low', 'medium', 'high'], palette=FAMILY_COLORS, ax=ax)
    sns.stripplot(data=spread_plot_df, x='family', y='pool_cost_spread_vs_milp_pct',
                  order=['low', 'medium', 'high'], color='black', alpha=0.25, size=3, ax=ax)
    ax.set_ylabel('Pool cost spread / |MILP| (%)', fontweight='bold')
else:
    ax.text(0.5, 0.5, 'No multi-candidate spread available yet', ha='center', va='center', transform=ax.transAxes)
    ax.set_axis_off()
ax.set_xlabel('Family', fontweight='bold')
ax.set_title('(A) Intra-Scenario Cost Spread', fontweight='bold')
ax.grid(axis='y', alpha=0.3)

ax = axes[1]
div_plot_df = diversity_df.dropna(subset=['binary_diversity_mean']).copy()
if len(div_plot_df):
    sns.boxplot(data=div_plot_df, x='family', y='binary_diversity_mean',
                order=['low', 'medium', 'high'], palette=FAMILY_COLORS, ax=ax)
    sns.stripplot(data=div_plot_df, x='family', y='binary_diversity_mean',
                  order=['low', 'medium', 'high'], color='black', alpha=0.25, size=3, ax=ax)
    ax.set_ylabel('Mean pairwise Hamming distance', fontweight='bold')
else:
    ax.text(0.5, 0.5,
            'Binary diversity needs a fresh run with n_samples > 1',
            ha='center', va='center', transform=ax.transAxes)
    ax.set_axis_off()
ax.set_xlabel('Family', fontweight='bold')
ax.set_title('(B) Binary Diversity over Sampled u', fontweight='bold')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_solution_diversity.png', dpi=300, bbox_inches='tight')
plt.show()

# --- Best-of-K figure --------------------------------------------------------
if len(bestofk_summary):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    ax = axes[0]
    ax.plot(bestofk_summary['k'], bestofk_summary['mean_cost_gap_pct'], marker='o', lw=2, color='#1f77b4')
    ax.axhline(0, color='red', linestyle='--', alpha=0.5)
    ax.set_xlabel('Candidate pool size K', fontweight='bold')
    ax.set_ylabel('Mean best-of-K cost gap (%)', fontweight='bold')
    ax.set_title('(A) Best-of-K Quality', fontweight='bold')
    ax.grid(True, alpha=0.3)

    ax = axes[1]
    if bestofk_summary['pct_beats_heuristic'].notna().any():
        ax.plot(bestofk_summary['k'], bestofk_summary['pct_beats_heuristic'], marker='o', lw=2, color='#2ca02c')
        ax.set_ylabel('% scenarios beating RH-MO+LP', fontweight='bold')
        ax.set_title('(B) Best-of-K vs Heuristic', fontweight='bold')
    else:
        ax.plot(bestofk_summary['k'], bestofk_summary['pct_pipeline_faster_than_milp'], marker='o', lw=2, color='#ff7f0e')
        ax.set_ylabel('% scenarios faster than MILP', fontweight='bold')
        ax.set_title('(B) Best-of-K Time Advantage', fontweight='bold')
    ax.set_xlabel('Candidate pool size K', fontweight='bold')
    ax.set_ylim(0, 105)
    ax.grid(True, alpha=0.3)

    ax = axes[2]
    ax.plot(bestofk_summary['k'], bestofk_summary['mean_total_time_s'], marker='o', lw=2, color='#d62728')
    ax.set_xlabel('Candidate pool size K', fontweight='bold')
    ax.set_ylabel('Mean actual wall-clock time (s)', fontweight='bold')
    ax.set_title('(C) Best-of-K Time Cost', fontweight='bold')
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'fig_best_of_k_curve.png', dpi=300, bbox_inches='tight')
    plt.show()


## 10b.1. K Samples vs Cost Gap

This subsection isolates the core sampling claim: as the candidate pool grows, the best feasible solution should move closer to the MILP reference.

We therefore track the **best-of-$K$ absolute cost gap** as a function of $K$, both globally and by family.


In [ ]:
if len(bestofk_scenario):
    scenario_abs_gap = bestofk_scenario.copy()
    scenario_abs_gap['abs_cost_gap_pct'] = scenario_abs_gap['cost_gap_pct'].abs()

    gap_curve = (
        scenario_abs_gap.groupby('k')
        .agg(
            mean_cost_gap_pct=('cost_gap_pct', 'mean'),
            median_cost_gap_pct=('cost_gap_pct', 'median'),
            mean_abs_cost_gap_pct=('abs_cost_gap_pct', 'mean'),
            median_abs_cost_gap_pct=('abs_cost_gap_pct', 'median'),
            p25_abs_cost_gap_pct=('abs_cost_gap_pct', lambda s: float(np.percentile(s.dropna(), 25)) if len(s.dropna()) else np.nan),
            p75_abs_cost_gap_pct=('abs_cost_gap_pct', lambda s: float(np.percentile(s.dropna(), 75)) if len(s.dropna()) else np.nan),
            n_scenarios=('scenario_id', 'nunique'),
        )
        .reset_index()
    )

    family_gap_curve = (
        scenario_abs_gap.groupby(['family', 'k'])
        .agg(
            mean_abs_cost_gap_pct=('abs_cost_gap_pct', 'mean'),
            median_abs_cost_gap_pct=('abs_cost_gap_pct', 'median'),
        )
        .reset_index()
    )

    k_pivot = scenario_abs_gap.pivot_table(
        index=['family', 'scenario_id'],
        columns='k',
        values='abs_cost_gap_pct',
        aggfunc='first',
    )
    baseline_k = min(BEST_OF_K_VALUES)
    max_k = max(BEST_OF_K_VALUES)
    pair_df = k_pivot[[baseline_k, max_k]].dropna() if baseline_k in k_pivot.columns and max_k in k_pivot.columns else pd.DataFrame()
    improvement_rate = float((pair_df[max_k] <= pair_df[baseline_k]).mean() * 100) if len(pair_df) else np.nan

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    ax = axes[0]
    ax.plot(gap_curve['k'], gap_curve['median_abs_cost_gap_pct'], marker='o', lw=2.5, color='#1f77b4', label='Median |gap|')
    ax.plot(gap_curve['k'], gap_curve['mean_abs_cost_gap_pct'], marker='s', lw=1.8, color='#ff7f0e', label='Mean |gap|')
    ax.fill_between(
        gap_curve['k'],
        gap_curve['p25_abs_cost_gap_pct'],
        gap_curve['p75_abs_cost_gap_pct'],
        alpha=0.18,
        color='#1f77b4',
        label='IQR |gap|',
    )
    ax.set_xlabel('Candidate pool size K', fontweight='bold')
    ax.set_ylabel('Best-of-K absolute cost gap vs MILP (%)', fontweight='bold')
    ax.set_title('(A) Overall K-Samples vs Cost Gap', fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9)

    ax = axes[1]
    for fam in ['low', 'medium', 'high']:
        fam_df = family_gap_curve[family_gap_curve['family'] == fam]
        if len(fam_df) == 0:
            continue
        ax.plot(
            fam_df['k'],
            fam_df['median_abs_cost_gap_pct'],
            marker='o',
            lw=2,
            label=fam.capitalize(),
            color=FAMILY_COLORS.get(fam),
        )
    ax.set_xlabel('Candidate pool size K', fontweight='bold')
    ax.set_ylabel('Median absolute cost gap vs MILP (%)', fontweight='bold')
    ax.set_title('(B) Family-Level Convergence with K', fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'fig_k_samples_vs_cost_gap.png', dpi=300, bbox_inches='tight')
    plt.show()

    print(f'Absolute-gap improvement from K={baseline_k} to K={max_k}: {improvement_rate:.1f}% of scenarios')
    display(gap_curve.round(3))
else:
    print('bestofk_scenario is empty: rerun Section 5 with n_samples > 1 to evaluate K-samples vs cost gap.')


## 10b.2. K-Sampling Diagnostics on the Full Run

For the full evaluation set, we compare the sampler at $K \in \{1,3,5,10\}$ through the exact checklist used in the smoke test:
- mean Hamming distance,
- activation rate,
- final EBM energy std,
- best-of-$K$ cost gap,
- slack and LP stage reached by the selected best-of-$K$ candidate.


In [ ]:
K_DIAG_VALUES = [k for k in [1, 3, 5, 10] if k <= MAX_AVAILABLE_SAMPLES]
if not K_DIAG_VALUES and MAX_AVAILABLE_SAMPLES:
    K_DIAG_VALUES = [MAX_AVAILABLE_SAMPLES]

kdiag_summary, kdiag_scenario = compute_k_sampling_diagnostics(
    pipeline_results,
    df,
    k_values=K_DIAG_VALUES or [1],
)

display(kdiag_summary.round(3))

if len(kdiag_scenario):
    family_kdiag_summary = (
        kdiag_scenario.groupby(['family', 'k'])
        .agg(
            mean_hamming=('prefix_mean_hamming', 'mean'),
            mean_activation_rate=('prefix_activation_rate_mean', 'mean'),
            mean_energy_std=('prefix_energy_std', 'mean'),
            median_best_cost_gap_pct=('best_of_k_cost_gap_pct', 'median'),
            median_best_abs_cost_gap_pct=('best_of_k_abs_cost_gap_pct', 'median'),
            median_best_slack_mwh=('best_of_k_slack_mwh', 'median'),
            n_scenarios=('scenario_id', 'nunique'),
        )
        .reset_index()
    )
    display(family_kdiag_summary.round(3))

    kdiag_stage_table = (
        kdiag_scenario.groupby(['k', 'best_of_k_stage_reached'])
        .size()
        .rename('count')
        .reset_index()
        .pivot(index='k', columns='best_of_k_stage_reached', values='count')
        .fillna(0)
    )
    display(kdiag_stage_table.astype(int))

    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    axes = axes.ravel()

    axes[0].plot(kdiag_summary['k'], kdiag_summary['mean_hamming'], marker='o', lw=2, color='#1f77b4')
    axes[0].set_title('(A) Mean Pairwise Hamming', fontweight='bold')
    axes[0].set_xlabel('K')
    axes[0].set_ylabel('Distance')
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(kdiag_summary['k'], kdiag_summary['mean_energy_std'], marker='o', lw=2, color='#d62728')
    axes[1].set_title('(B) Final EBM Energy Std', fontweight='bold')
    axes[1].set_xlabel('K')
    axes[1].set_ylabel('Std(E)')
    axes[1].grid(True, alpha=0.3)

    axes[2].plot(kdiag_summary['k'], kdiag_summary['mean_activation_rate'], marker='o', lw=2, color='#2ca02c')
    axes[2].set_title('(C) Mean Activation Rate', fontweight='bold')
    axes[2].set_xlabel('K')
    axes[2].set_ylabel('Active fraction')
    axes[2].grid(True, alpha=0.3)

    axes[3].plot(kdiag_summary['k'], kdiag_summary['median_best_abs_cost_gap_pct'], marker='o', lw=2.5, color='black', label='Overall median |gap|')
    for fam in ['low', 'medium', 'high']:
        fam_df = family_kdiag_summary[family_kdiag_summary['family'] == fam]
        if len(fam_df) == 0:
            continue
        axes[3].plot(fam_df['k'], fam_df['median_best_abs_cost_gap_pct'], marker='o', lw=1.6, color=FAMILY_COLORS.get(fam), label=fam.capitalize())
    axes[3].set_title('(D) Best-of-K Absolute Cost Gap', fontweight='bold')
    axes[3].set_xlabel('K')
    axes[3].set_ylabel('|Cost gap| vs MILP (%)')
    axes[3].grid(True, alpha=0.3)
    axes[3].legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'fig_k_sampling_diagnostics_fullrun.png', dpi=300, bbox_inches='tight')
    plt.show()

    base_k = min(K_DIAG_VALUES)
    top_k = max(K_DIAG_VALUES)
    pivot_abs_gap = kdiag_scenario.pivot_table(
        index=['family', 'scenario_id'],
        columns='k',
        values='best_of_k_abs_cost_gap_pct',
        aggfunc='first',
    )
    if base_k in pivot_abs_gap.columns and top_k in pivot_abs_gap.columns:
        paired = pivot_abs_gap[[base_k, top_k]].dropna()
        improvement_rate = float((paired[top_k] <= paired[base_k]).mean() * 100) if len(paired) else np.nan
        print(f'Absolute cost gap improved from K={base_k} to K={top_k} on {improvement_rate:.1f}% of full-run scenarios.')

    kdiag_summary.to_csv(OUTPUT_DIR / 'k_sampling_diagnostics_summary.csv', index=False)
    family_kdiag_summary.to_csv(OUTPUT_DIR / 'k_sampling_diagnostics_by_family.csv', index=False)
    kdiag_scenario.to_csv(OUTPUT_DIR / 'k_sampling_diagnostics_per_scenario.csv', index=False)
else:
    print('K-sampling diagnostics unavailable: rerun Section 5 with n_samples >= 2 (ideally 10).')


## 10c. Complexity Scaling Law

We test the paper-facing claim that the hybrid pipeline gains **relative value as criticality increases**.

Main figure:
- Panel A: $\log(S_i) = lpha + eta C_i + arepsilon_i$ with $S_i = T_i^{MILP} / T_i^{pipe}$,
- Panel B: $P(S_i > 1)$ via logistic fit,
- Panel C: $|	ext{cost gap}|$ vs criticality, to verify that quality does not diverge.

As a robustness check against the composite criticality index, we also fit the same story with physical covariates: number of zones, number of binaries, peak-to-valley ratio, storage adequacy, and VRE volatility.


In [ ]:
from src.eval.advanced_analysis import (
    PHYSICAL_FEATURE_COLUMNS,
    build_physical_complexity_frame,
    merge_physical_complexity_features,
    fit_scaling_law_models,
    fit_physical_feature_robustness,
)

physical_feature_df = build_physical_complexity_frame(
    {name: paths['scenarios_dir'] for name, paths in FAMILIES.items()},
    cache_path=OUTPUT_DIR / 'physical_complexity_features.csv',
)

df_scaling = merge_physical_complexity_features(df, physical_feature_df)
scaling_models = fit_scaling_law_models(
    df_scaling,
    feature_col='criticality_index',
    cost_gap_col='cost_gap_pct',
    use_abs_gap=True,
)
physical_robustness_df = fit_physical_feature_robustness(
    df_scaling,
    feature_cols=PHYSICAL_FEATURE_COLUMNS,
    cost_gap_col='cost_gap_pct',
    use_abs_gap=True,
)

print('Scaling-law summary:')
for key, value in scaling_models.get('summary', {}).items():
    if isinstance(value, float):
        print(f'  {key}: {value:.4g}')
    else:
        print(f'  {key}: {value}')

display(physical_robustness_df)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
summary = scaling_models['summary']

# --- Panel A: log-speedup vs criticality ------------------------------------
ax = axes[0]
pts = scaling_models['panel_a_points']
fit = scaling_models['panel_a_fit']
ax.scatter(pts['criticality_index'], pts['speedup'], alpha=0.45, s=28, color='#1f77b4', edgecolors='white', linewidths=0.3)
ax.plot(fit['criticality_index'], np.exp(fit['pred_log_speedup']), color='black', lw=2)
ax.axhline(1.0, color='red', linestyle='--', alpha=0.4)
ax.set_yscale('log')
ax.set_xlabel('Criticality index', fontweight='bold')
ax.set_ylabel('Speedup vs MILP (x, log scale)', fontweight='bold')
ax.set_title(f"(A) log-speedup beta={summary['panel_a_beta']:.2f}", fontweight='bold')
ax.grid(True, alpha=0.3)

# --- Panel B: probability pipeline faster -----------------------------------
ax = axes[1]
pts = scaling_models['panel_b_points'].copy()
fit = scaling_models['panel_b_fit']
if len(pts):
    jitter = np.random.default_rng(42).uniform(-0.03, 0.03, size=len(pts))
    ax.scatter(pts['criticality_index'], pts['pipeline_faster'] + jitter,
               alpha=0.2, s=18, color='#2ca02c')
if len(fit):
    ax.plot(fit['criticality_index'], fit['prob_pipeline_faster'], color='black', lw=2)
ax.set_xlabel('Criticality index', fontweight='bold')
ax.set_ylabel('P(speedup > 1)', fontweight='bold')
ax.set_title(f"(B) logistic beta={summary['panel_b_beta']:.2f}", fontweight='bold')
ax.set_ylim(-0.05, 1.05)
ax.grid(True, alpha=0.3)

# --- Panel C: bounded cost-gap magnitude ------------------------------------
ax = axes[2]
pts = scaling_models['panel_c_points']
fit = scaling_models['panel_c_fit']
ax.scatter(pts['criticality_index'], pts['cost_gap_target'], alpha=0.45, s=28,
           color='#d62728', edgecolors='white', linewidths=0.3)
ax.plot(fit['criticality_index'], fit['pred_cost_gap_target'], color='black', lw=2)
ax.set_xlabel('Criticality index', fontweight='bold')
ax.set_ylabel('|Cost gap| (%)', fontweight='bold')
ax.set_title(f"(C) gap beta={summary['panel_c_beta']:.2f}", fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_scaling_law.png', dpi=300, bbox_inches='tight')
plt.show()

if not physical_robustness_df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
    coef_specs = [
        ('beta_log_speedup', '(A) Physical drivers of log-speedup'),
        ('beta_pipeline_faster_logit', '(B) Physical drivers of P(speedup > 1)'),
        ('beta_cost_gap_target', '(C) Physical drivers of |cost gap|'),
    ]
    for ax, (col, title) in zip(axes, coef_specs):
        order = physical_robustness_df.sort_values(col)[['feature', col]]
        ax.barh(order['feature'], order[col], color='#4c72b0')
        ax.axvline(0, color='black', lw=1)
        ax.set_title(title, fontweight='bold')
        ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'fig_scaling_physical_robustness.png', dpi=300, bbox_inches='tight')
    plt.show()


## 11. Economic Decision Threshold

The economic indicator is interpreted as a **decision-theoretic threshold**, not as a universal monetary gain.

**Formula:**
$$	ext{Total Cost}_{	ext{method}} = C_{	ext{solution}} + \lambda \cdot T_{	ext{solve}}$$

$$	ext{Advantage} = 	ext{Total Cost}_{	ext{MILP}} - 	ext{Total Cost}_{	ext{Pipeline}}$$

At breakeven,
$$\lambda_i^* = rac{C_i^{	ext{pipe}} - C_i^{	ext{MILP}}}{T_i^{	ext{MILP}} - T_i^{	ext{pipe}}}$$

so the pipeline is preferable when the time saved compensates for the additional operating cost.


In [ ]:
from src.eval.economic_advantage import EconomicAdvantageAnalyzer

econ = EconomicAdvantageAnalyzer(df)

# Global sensitivity analysis
sensitivity_df = econ.sensitivity_analysis()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Panel A: % preferred vs lambda -----------------------------------------
ax = axes[0]
ax.plot(sensitivity_df['lambda'], sensitivity_df['pct_profitable'], 'b-', lw=2)
ax.axhline(y=50, color='red', linestyle=':', alpha=0.5, label='50%')
ax.axhline(y=80, color='orange', linestyle=':', alpha=0.5, label='80%')
ax.set_xlabel('Lambda (EUR/second)', fontweight='bold')
ax.set_ylabel('% scenarios preferring pipeline', fontweight='bold')
ax.set_title('(A) Preference Rate vs Time Value')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 105)

# --- Panel B: mean advantage vs lambda --------------------------------------
ax = axes[1]
ax.plot(sensitivity_df['lambda'], sensitivity_df['mean_advantage'] / 1e6, 'g-', lw=2)
ax.axhline(y=0, color='red', linestyle='--', alpha=0.5)
ax.set_xlabel('Lambda (EUR/second)', fontweight='bold')
ax.set_ylabel('Mean advantage (M EUR)', fontweight='bold')
ax.set_title('(B) Mean Decision Advantage')
ax.grid(True, alpha=0.3)

# --- Panel C: by family ------------------------------------------------------
ax = axes[2]
family_sens = econ.sensitivity_by_family()
for fam_name, fam_df in family_sens.items():
    color = FAMILY_COLORS.get(fam_name, 'gray')
    ax.plot(fam_df['lambda'], fam_df['pct_profitable'], '-', color=color, lw=2, label=fam_name)
ax.axhline(y=50, color='gray', linestyle=':', alpha=0.3)
ax.set_xlabel('Lambda (EUR/second)', fontweight='bold')
ax.set_ylabel('% scenarios preferring pipeline', fontweight='bold')
ax.set_title('(C) Preference by Family')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 105)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig4_economic_advantage.png', dpi=300, bbox_inches='tight')
plt.show()


## 12. Breakeven Lambda Analysis

In [ ]:
# Find breakeven lambda values
summary = econ.compute_summary(lambda_values=[1, 5, 10, 20, 50, 100])

print('='*60)
print('DECISION-THRESHOLD SUMMARY')
print('='*60)
print(f"Breakeven lambda (50% preferring pipeline): {summary['breakeven_lambda_50pct']:.2f} EUR/s")
print(f"Breakeven lambda (80% preferring pipeline): {summary['breakeven_lambda_80pct']:.2f} EUR/s")
print(f"Breakeven lambda (95% preferring pipeline): {summary['breakeven_lambda_95pct']:.2f} EUR/s")

print(f"\nPer-family breakeven (50%):")
for fam, lam in summary['family_breakevens'].items():
    print(f"  {fam}: {lam:.2f} EUR/s")

print(f"\nAt selected lambda values:")
for key, val in summary.items():
    if key.startswith('lambda_'):
        lam = key.replace('lambda_', '')
        print(f"  lambda={lam} EUR/s: {val['pct_profitable']:.1f}% preferring pipeline, "
              f"mean advantage={val['mean_advantage_eur']/1e6:.2f} MEUR")


## 12b. Scenario Breakeven Lambda and Decision Regimes

Three regimes are tracked explicitly:
- **offline planning**: low $\lambda$,
- **operational screening**: intermediate $\lambda$,
- **stress / emergency assessment**: high $\lambda$.

We complement the global sensitivity curve with per-scenario breakeven thresholds and the rule associated with each case (`pipeline_always`, `pipeline_if_lambda_gt_lambda_star`, `pipeline_if_lambda_lt_lambda_star`, `milp_always`).


In [ ]:
breakeven_df = econ.compute_breakeven_lambda_per_scenario()
regime_df = econ.summarize_decision_regimes(
    regime_lambdas={
        'offline': 0.5,
        'screening': 5.0,
        'emergency': 25.0,
    }
)

display(breakeven_df.head())
display(regime_df)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Panel A: lambda* distribution when pipeline is faster but costlier -----
ax = axes[0]
lambda_star_df = breakeven_df[
    breakeven_df['lambda_star_rule'] == 'pipeline_if_lambda_gt_lambda_star'
].copy()
lambda_star_df = lambda_star_df[np.isfinite(lambda_star_df['lambda_star_eur_per_s'])]
if len(lambda_star_df):
    sns.boxplot(data=lambda_star_df, x='family', y='lambda_star_eur_per_s',
                order=['low', 'medium', 'high'], palette=FAMILY_COLORS, ax=ax)
    sns.stripplot(data=lambda_star_df, x='family', y='lambda_star_eur_per_s',
                  order=['low', 'medium', 'high'], color='black', alpha=0.25, size=3, ax=ax)
    ax.set_yscale('log')
    ax.set_ylabel('Breakeven lambda* (EUR/s)', fontweight='bold')
else:
    ax.text(0.5, 0.5, 'No faster-but-costlier cases in current results',
            ha='center', va='center', transform=ax.transAxes)
    ax.set_axis_off()
ax.set_xlabel('Family', fontweight='bold')
ax.set_title('(A) Lambda* Distribution', fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# --- Panel B: decision-rule mix ---------------------------------------------
ax = axes[1]
rule_order = [
    'pipeline_always',
    'pipeline_if_lambda_gt_lambda_star',
    'pipeline_if_lambda_lt_lambda_star',
    'milp_always',
]
rule_colors = {
    'pipeline_always': '#2ca02c',
    'pipeline_if_lambda_gt_lambda_star': '#1f77b4',
    'pipeline_if_lambda_lt_lambda_star': '#ff7f0e',
    'milp_always': '#d62728',
}
family_order = ['low', 'medium', 'high']
for fam_idx, fam in enumerate(family_order):
    fam_df = breakeven_df[breakeven_df['family'] == fam]
    total = max(len(fam_df), 1)
    bottom = 0.0
    for rule in rule_order:
        pct = 100.0 * (fam_df['lambda_star_rule'] == rule).sum() / total
        ax.bar(fam_idx, pct, bottom=bottom, color=rule_colors[rule], width=0.65,
               label=rule if fam_idx == 0 else '')
        if pct >= 6:
            ax.text(fam_idx, bottom + pct / 2, f'{pct:.0f}%', ha='center', va='center', fontsize=8, color='white')
        bottom += pct
ax.set_xticks(range(len(family_order)))
ax.set_xticklabels([fam.title() for fam in family_order])
ax.set_ylabel('Share of scenarios (%)', fontweight='bold')
ax.set_title('(B) Decision Rule Mix', fontweight='bold')
ax.legend(fontsize=8)
ax.set_ylim(0, 105)
ax.grid(axis='y', alpha=0.3)

# --- Panel C: pipeline preference by lambda regime ---------------------------
ax = axes[2]
for fam in family_order:
    fam_df = regime_df[regime_df['family'] == fam]
    ax.plot(fam_df['regime'], fam_df['pct_pipeline_preferred'], marker='o', lw=2,
            color=FAMILY_COLORS[fam], label=fam)
ax.set_ylabel('% scenarios preferring pipeline', fontweight='bold')
ax.set_title('(C) Decision Regimes', fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend()
ax.set_ylim(0, 105)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_breakeven_lambda_distribution.png', dpi=300, bbox_inches='tight')
plt.show()


## 13. Lambda Sensitivity Heatmap

In [ ]:
lambda_values = [0.5, 1, 2, 5, 10, 20, 50, 100]
families_list = ['low', 'medium', 'high']

# Build heatmap data: rows=families, cols=lambda, values=pipeline preference rate
heat_data = np.zeros((len(families_list), len(lambda_values)))
for i, fam in enumerate(families_list):
    sub_df = df[df['family'] == fam]
    sub_econ = EconomicAdvantageAnalyzer(sub_df)
    for j, lam in enumerate(lambda_values):
        adv_df = sub_econ.compute_advantage(lam)
        valid = adv_df.dropna(subset=['advantage'])
        heat_data[i, j] = valid['pipeline_profitable'].mean() * 100

fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(heat_data, cmap='RdYlGn', aspect='auto', vmin=0, vmax=100)

ax.set_xticks(range(len(lambda_values)))
ax.set_xticklabels([f'{l}' for l in lambda_values])
ax.set_yticks(range(len(families_list)))
ax.set_yticklabels(['Low Crit.', 'Medium Crit.', 'High Crit.'])
ax.set_xlabel('Lambda (EUR/second)', fontweight='bold')
ax.set_title('% Pipeline Preferred: Family x Lambda', fontsize=14, fontweight='bold')

# Annotate cells
for i in range(len(families_list)):
    for j in range(len(lambda_values)):
        val = heat_data[i, j]
        color = 'white' if val < 30 or val > 70 else 'black'
        ax.text(j, i, f'{val:.0f}%', ha='center', va='center', color=color, fontweight='bold')

plt.colorbar(im, ax=ax, label='% preferring pipeline')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig5_lambda_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()


## 14. Cumulative Time Savings

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, fam in enumerate(families_list):
    ax = axes[idx]
    fam_df = df[df['family'] == fam].sort_values('criticality_index').reset_index(drop=True)

    cum_milp = fam_df['milp_solve_time'].cumsum() / 3600
    cum_pipe = fam_df['pipeline_solve_time'].cumsum() / 3600

    ax.fill_between(range(len(fam_df)), cum_milp, alpha=0.3, color='red', label='MILP')
    ax.fill_between(range(len(fam_df)), cum_pipe, alpha=0.3, color='green', label='Pipeline')
    ax.plot(range(len(fam_df)), cum_milp, 'r-', lw=2)
    ax.plot(range(len(fam_df)), cum_pipe, 'g-', lw=2)

    total_milp = cum_milp.iloc[-1] if len(cum_milp) > 0 else 0
    total_pipe = cum_pipe.iloc[-1] if len(cum_pipe) > 0 else 0
    saved = total_milp - total_pipe
    pct_saved = saved / max(total_milp, 1e-9) * 100

    ax.set_xlabel('Scenarios (sorted by criticality)')
    ax.set_ylabel('Cumulative Time (hours)')
    ax.set_title(f'{fam.capitalize()} Criticality\nSaved: {saved:.1f}h ({pct_saved:.0f}%)')
    ax.legend(loc='upper left')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig6_cumulative_time.png', dpi=300, bbox_inches='tight')
plt.show()

## 15. Pipeline Timing Breakdown

In [ ]:
timing_cols = ['time_graph_build', 'time_embedding', 'time_ebm_sampling', 'time_decoder', 'time_lp_solve']
timing_labels = ['Graph Build', 'HTE Embedding', 'EBM Sampling', 'Warm-Start', 'LP Solve']
timing_colors = ['#3498db', '#2ecc71', '#e74c3c', '#f1c40f', '#9b59b6']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, fam in enumerate(families_list):
    ax = axes[idx]
    fam_df = df[df['family'] == fam]
    means = [fam_df[col].mean() for col in timing_cols]

    ax.barh(timing_labels, means, color=timing_colors, alpha=0.85)
    for i, v in enumerate(means):
        ax.text(v + 0.1, i, f'{v:.1f}s', va='center', fontsize=9)
    ax.set_xlabel('Time (seconds)')
    ax.set_title(f'{fam.capitalize()} Criticality')
    ax.grid(True, alpha=0.3, axis='x')

fig.suptitle('Pipeline Timing Breakdown by Family', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig7_timing_breakdown.png', dpi=300, bbox_inches='tight')
plt.show()

## 16. Criticality Index Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Panel A: Criticality Distribution ---
ax = axes[0]
for fam, color in FAMILY_COLORS.items():
    fam_df = df[df['family'] == fam]
    ax.hist(fam_df['criticality_index'], bins=20, alpha=0.5, color=color,
            label=f'{fam} (n={len(fam_df)})', edgecolor='white')
ax.set_xlabel('Criticality Index')
ax.set_ylabel('Count')
ax.set_title('(A) Criticality Index Distribution')
ax.legend()

# --- Panel B: Complexity (n_zones * n_timesteps) ---
ax = axes[1]
df['complexity'] = df['n_zones'] * df['n_timesteps']
for fam, color in FAMILY_COLORS.items():
    mask = df['family'] == fam
    ax.scatter(df.loc[mask, 'complexity'], df.loc[mask, 'speedup'],
              c=color, label=fam, s=40, alpha=0.7)
ax.set_xlabel('Complexity (zones x timesteps)')
ax.set_ylabel('Speedup (x)')
ax.set_title('(B) Speedup vs Problem Complexity')
ax.set_yscale('log')
ax.legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig8_criticality_complexity.png', dpi=300, bbox_inches='tight')
plt.show()

## 17. Final Summary Table

In [ ]:
# Build summary table - combined Pipeline + Heuristic + MILP reference
PIPE_STAGE_COL = 'pipeline_stage_reached' if 'pipeline_stage_reached' in df.columns else 'pipeline_stage'
HEUR_STAGE_COL = 'heur_stage_reached' if 'heur_stage_reached' in df.columns else 'heur_stage'


def _stage_reached_pct(sub_df, reached_col, fallback_stage_col, stage_name, include_later=False):
    if reached_col in sub_df.columns:
        return sub_df[reached_col].fillna(False).astype(bool).mean() * 100
    if fallback_stage_col in sub_df.columns:
        if include_later and stage_name == 'full_soft':
            return sub_df[fallback_stage_col].isin(['full_soft', 'round_refix']).mean() * 100
        return (sub_df[fallback_stage_col] == stage_name).mean() * 100
    return 0.0


def _build_row(name, sub_df):
    valid_p = sub_df.dropna(subset=['speedup', 'cost_gap_pct'])
    valid_h = sub_df.dropna(subset=['heur_speedup', 'heur_cost_gap_pct']) if HAS_HEURISTIC else sub_df.iloc[0:0]
    row = {
        'Family': name,
        'N': len(sub_df),
        'Crit. Index': f"{sub_df['criticality_index'].mean():.3f}",
        # --- Pipeline (GNN+EBM) ---
        'Pipe Success':       f"{sub_df['success'].mean()*100:.0f}%",
        'Pipe Speedup (med)': f"{valid_p['speedup'].median():.1f}x"      if len(valid_p) else 'N/A',
        'Pipe Gap P50':       f"{valid_p['cost_gap_pct'].median():.1f}%" if len(valid_p) else 'N/A',
        'Pipe Gap P90':       f"{np.percentile(valid_p['cost_gap_pct'], 90):.1f}%" if len(valid_p) else 'N/A',
        'Pipe Gap P99':       f"{np.percentile(valid_p['cost_gap_pct'], 99):.1f}%" if len(valid_p) else 'N/A',
        'Pipe Time (mean)':   f"{sub_df['pipeline_solve_time'].mean():.1f}s",
        'Pipe HardFix %':     f"{_stage_reached_pct(sub_df, 'pipeline_reached_hard_fix', PIPE_STAGE_COL, 'hard_fix'):.0f}%",
        'Pipe S4 reached %':  f"{_stage_reached_pct(sub_df, 'pipeline_reached_full_soft', PIPE_STAGE_COL, 'full_soft', include_later=True):.0f}%",
        'Pipe S5 reached %':  f"{_stage_reached_pct(sub_df, 'pipeline_reached_round_refix', PIPE_STAGE_COL, 'round_refix'):.0f}%",
        # --- Heuristic (RH-MO+LP) ---
        'Heur Success':       f"{sub_df['heur_success'].mean()*100:.0f}%" if HAS_HEURISTIC and sub_df['heur_success'].notna().any() else 'N/A',
        'Heur Speedup (med)': f"{valid_h['heur_speedup'].median():.1f}x" if len(valid_h) else 'N/A',
        'Heur Gap P50':       f"{valid_h['heur_cost_gap_pct'].median():.1f}%" if len(valid_h) else 'N/A',
        'Heur Gap P90':       f"{np.percentile(valid_h['heur_cost_gap_pct'], 90):.1f}%" if len(valid_h) else 'N/A',
        'Heur Gap P99':       f"{np.percentile(valid_h['heur_cost_gap_pct'], 99):.1f}%" if len(valid_h) else 'N/A',
        'Heur Time (mean)':   f"{sub_df['heur_solve_time'].mean():.1f}s" if HAS_HEURISTIC and sub_df['heur_solve_time'].notna().any() else 'N/A',
        'Heur S4 reached %':  f"{_stage_reached_pct(sub_df, 'heur_reached_full_soft', HEUR_STAGE_COL, 'full_soft', include_later=True):.0f}%" if HAS_HEURISTIC else 'N/A',
        'Heur S5 reached %':  f"{_stage_reached_pct(sub_df, 'heur_reached_round_refix', HEUR_STAGE_COL, 'round_refix'):.0f}%" if HAS_HEURISTIC else 'N/A',
        # --- MILP reference ---
        'MILP Time (mean)':   f"{sub_df['milp_solve_time'].mean():.1f}s",
    }
    return row

summary_rows = [_build_row(fam.capitalize(), df[df['family'] == fam]) for fam in families_list]
summary_rows.append(_build_row('ALL', df))

summary_table = pd.DataFrame(summary_rows)
print('\n' + '=' * 150)
print('FINAL EVALUATION SUMMARY: Pipeline (GNN+EBM) vs Heuristic (RH-MO+LP) vs MILP')
print('=' * 150)
with pd.option_context('display.max_columns', None, 'display.width', 260):
    print(summary_table.to_string(index=False))

# Save
summary_table.to_csv(OUTPUT_DIR / 'summary_table.csv', index=False)
df.to_csv(OUTPUT_DIR / 'full_comparison.csv', index=False)
print(f'\nResults saved to {OUTPUT_DIR}')


In [ ]:
# --- Extended summary table INCLUDING heuristic baseline -------------------
if not HAS_HEURISTIC:
    print('Heuristic results not loaded - skipping extended summary.')
else:
    PIPE_STAGE_COL = 'pipeline_stage_reached' if 'pipeline_stage_reached' in df.columns else 'pipeline_stage'
    HEUR_STAGE_COL = 'heur_stage_reached' if 'heur_stage_reached' in df.columns else 'heur_stage'

    def _stage_reached_pct(sub_df, reached_col, fallback_stage_col, stage_name, include_later=False):
        if reached_col in sub_df.columns:
            return sub_df[reached_col].fillna(False).astype(bool).mean() * 100
        if fallback_stage_col in sub_df.columns:
            if include_later and stage_name == 'full_soft':
                return sub_df[fallback_stage_col].isin(['full_soft', 'round_refix']).mean() * 100
            return (sub_df[fallback_stage_col] == stage_name).mean() * 100
        return 0.0

    ext_rows = []
    def _row(name, sub):
        valid_p = sub.dropna(subset=['speedup', 'cost_gap_pct'])
        valid_h = sub.dropna(subset=['heur_speedup', 'heur_cost_gap_pct'])
        return {
            'Family': name,
            'N': len(sub),
            # --- Pipeline ---
            'Pipe Speedup (med)': f"{valid_p['speedup'].median():.1f}x" if len(valid_p) else 'N/A',
            'Pipe Gap P50':       f"{valid_p['cost_gap_pct'].median():.1f}%" if len(valid_p) else 'N/A',
            'Pipe Gap P90':       f"{np.percentile(valid_p['cost_gap_pct'], 90):.1f}%" if len(valid_p) else 'N/A',
            'Pipe Time (mean)':   f"{sub['pipeline_solve_time'].mean():.1f}s",
            'Pipe S4 reached %':  f"{_stage_reached_pct(sub, 'pipeline_reached_full_soft', PIPE_STAGE_COL, 'full_soft', include_later=True):.0f}%",
            'Pipe S5 reached %':  f"{_stage_reached_pct(sub, 'pipeline_reached_round_refix', PIPE_STAGE_COL, 'round_refix'):.0f}%",
            # --- Heuristic ---
            'Heur Speedup (med)': f"{valid_h['heur_speedup'].median():.1f}x" if len(valid_h) else 'N/A',
            'Heur Gap P50':       f"{valid_h['heur_cost_gap_pct'].median():.1f}%" if len(valid_h) else 'N/A',
            'Heur Gap P90':       f"{np.percentile(valid_h['heur_cost_gap_pct'], 90):.1f}%" if len(valid_h) else 'N/A',
            'Heur Time (mean)':   f"{sub['heur_solve_time'].mean():.1f}s" if sub['heur_solve_time'].notna().any() else 'N/A',
            'Heur Success':       f"{sub['heur_success'].mean()*100:.0f}%" if sub['heur_success'].notna().any() else 'N/A',
            'Heur S4 reached %':  f"{_stage_reached_pct(sub, 'heur_reached_full_soft', HEUR_STAGE_COL, 'full_soft', include_later=True):.0f}%",
            'Heur S5 reached %':  f"{_stage_reached_pct(sub, 'heur_reached_round_refix', HEUR_STAGE_COL, 'round_refix'):.0f}%",
            # --- MILP reference ---
            'MILP Time (mean)':   f"{sub['milp_solve_time'].mean():.1f}s",
        }

    for fam in families_list:
        ext_rows.append(_row(fam.capitalize(), df[df['family'] == fam]))
    ext_rows.append(_row('ALL', df))

    ext_table = pd.DataFrame(ext_rows)
    print('\n' + '=' * 150)
    print('EXTENDED SUMMARY: Pipeline (GNN+EBM) vs Heuristic (RH-MO+LP) vs MILP')
    print('=' * 150)
    print(ext_table.to_string(index=False))

    ext_table.to_csv(OUTPUT_DIR / 'summary_table_with_heuristic.csv', index=False)
    print(f'\nExtended summary saved to {OUTPUT_DIR / "summary_table_with_heuristic.csv"}')


## 18. Save All Figures and Data

In [ ]:
# Save economic analysis results
econ_summary = econ.compute_summary(lambda_values=[0.5, 1, 2, 5, 10, 20, 50, 100])
with open(OUTPUT_DIR / 'economic_summary.json', 'w') as f:
    json.dump(econ_summary, f, indent=2, default=str)

# Save sensitivity data
sensitivity_df.to_csv(OUTPUT_DIR / 'lambda_sensitivity.csv', index=False)

# Save per-family sensitivity
for fam_name, fam_sens_df in family_sens.items():
    fam_sens_df.to_csv(OUTPUT_DIR / f'lambda_sensitivity_{fam_name}.csv', index=False)

# Save full metrics
eval_metrics_serializable = {
    k: v for k, v in eval_output.items() if k != 'dataframe'
}
with open(OUTPUT_DIR / 'eval_metrics.json', 'w') as f:
    json.dump(eval_metrics_serializable, f, indent=2, default=str)

# Save advanced analysis tables
if 'candidate_pool_df' in globals():
    candidate_pool_df.to_csv(OUTPUT_DIR / 'candidate_pool_samples.csv', index=False)
if 'diversity_df' in globals():
    diversity_df.to_csv(OUTPUT_DIR / 'solution_diversity.csv', index=False)
if 'bestofk_summary' in globals() and len(bestofk_summary):
    bestofk_summary.to_csv(OUTPUT_DIR / 'best_of_k_summary.csv', index=False)
if 'bestofk_scenario' in globals() and len(bestofk_scenario):
    bestofk_scenario.to_csv(OUTPUT_DIR / 'best_of_k_per_scenario.csv', index=False)
if 'physical_feature_df' in globals():
    physical_feature_df.to_csv(OUTPUT_DIR / 'physical_complexity_features.csv', index=False)
if 'physical_robustness_df' in globals() and len(physical_robustness_df):
    physical_robustness_df.to_csv(OUTPUT_DIR / 'physical_feature_robustness.csv', index=False)
if 'scaling_models' in globals():
    with open(OUTPUT_DIR / 'scaling_law_summary.json', 'w') as f:
        json.dump(scaling_models.get('summary', {}), f, indent=2, default=str)
if 'breakeven_df' in globals():
    breakeven_df.to_csv(OUTPUT_DIR / 'breakeven_lambda_per_scenario.csv', index=False)
if 'regime_df' in globals():
    regime_df.to_csv(OUTPUT_DIR / 'decision_regimes_summary.csv', index=False)

print(f'All outputs saved to: {OUTPUT_DIR}')
print('
Files:')
for f in sorted(OUTPUT_DIR.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name}: {size_kb:.1f} KB')


## 19. Focused Analysis: High-Criticality MaxTimeLimit Scenarios

The pipeline's value proposition is strongest on MILP-hard scenarios that hit the solver time limit.
This section isolates the **high-criticality MaxTimeLimit** subset and computes:
1. Corrected comparison DataFrame (with family-aware MILP matching)
2. Speed analysis
3. Cost gap analysis
4. Economic advantage with lambda sensitivity

In [ ]:
# --- Rebuild DataFrame with corrected MILP matching (family-prefixed keys) ---
from src.eval.metrics import compute_eval_metrics, format_metrics_table

milp_reports_dirs = {name: paths['reports_dir'] for name, paths in FAMILIES.items()}
eval_output = compute_eval_metrics(pipeline_results, milp_reports_dirs)
df = eval_output['dataframe']

# Sanity check: MILP times should now differ across families
print("MILP solve time per family (sanity check):")
for fam in ['low', 'medium', 'high']:
    fam_df = df[df['family'] == fam]
    print(f"  {fam}: MILP mean={fam_df['milp_solve_time'].mean():.1f}s, "
          f"Pipeline mean={fam_df['pipeline_solve_time'].mean():.1f}s")

# Filter: High-criticality + MaxTimeLimit only
df_htl = df[(df['family'] == 'high') & (df['milp_termination'] == 'maxTimeLimit')].copy()
print(f"\nHigh-crit MaxTimeLimit scenarios: {len(df_htl)} / {len(df[df['family'] == 'high'])} high-crit total")
print(f"Pipeline solve time: mean={df_htl['pipeline_solve_time'].mean():.1f}s, "
      f"median={df_htl['pipeline_solve_time'].median():.1f}s")
print(f"MILP solve time (all ~1200s): mean={df_htl['milp_solve_time'].mean():.1f}s")
print(f"Speedup: mean={df_htl['speedup'].mean():.1f}x, median={df_htl['speedup'].median():.1f}x")

In [ ]:
# --- Speed Analysis: High-Crit MaxTimeLimit ---
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel A: Pipeline vs MILP solve time (scatter)
ax = axes[0]
ax.scatter(df_htl['milp_solve_time'], df_htl['pipeline_solve_time'],
           c='#e74c3c', s=60, alpha=0.7, edgecolors='white', linewidth=0.5)
ax.plot([0, 1300], [0, 1300], 'k--', alpha=0.3, label='x = y')
ax.axhline(y=df_htl['pipeline_solve_time'].median(), color='blue',
           linestyle=':', alpha=0.6, label=f"Pipeline median={df_htl['pipeline_solve_time'].median():.0f}s")
ax.set_xlabel('MILP Solve Time (s)')
ax.set_ylabel('Pipeline Solve Time (s)')
ax.set_title(f'(A) Solve Time Comparison (n={len(df_htl)})')
ax.legend(fontsize=9)
ax.set_xlim(0, 1300)
ax.grid(True, alpha=0.3)

# Panel B: Speedup distribution
ax = axes[1]
ax.hist(df_htl['speedup'], bins=20, color='#e74c3c', alpha=0.7, edgecolor='white')
ax.axvline(x=df_htl['speedup'].median(), color='blue', linestyle='--', lw=2,
           label=f"Median={df_htl['speedup'].median():.1f}x")
ax.axvline(x=df_htl['speedup'].mean(), color='green', linestyle='--', lw=2,
           label=f"Mean={df_htl['speedup'].mean():.1f}x")
ax.axvline(x=1, color='red', linestyle=':', alpha=0.5, label='Break-even')
ax.set_xlabel('Speedup (MILP time / Pipeline time)')
ax.set_ylabel('Count')
ax.set_title('(B) Speedup Distribution')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Panel C: Pipeline time breakdown (stacked bar)
ax = axes[2]
timing_cols = ['time_graph_build', 'time_embedding', 'time_ebm_sampling', 'time_decoder', 'time_lp_solve']
timing_labels = ['Graph Build', 'HTE Embed', 'EBM Sample', 'Warm-Start', 'LP Solve']
timing_colors = ['#3498db', '#2ecc71', '#e74c3c', '#f1c40f', '#9b59b6']
means = [df_htl[col].mean() for col in timing_cols]
bars = ax.barh(timing_labels, means, color=timing_colors, alpha=0.85)
for bar, v in zip(bars, means):
    ax.text(v + 1, bar.get_y() + bar.get_height()/2, f'{v:.1f}s', va='center', fontsize=9)
ax.set_xlabel('Time (seconds)')
ax.set_title(f'(C) Pipeline Timing Breakdown (mean)')
ax.grid(True, alpha=0.3, axis='x')

fig.suptitle('Speed Analysis: High-Criticality MaxTimeLimit Scenarios', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig9_htl_speed_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

# Print stats
print(f"Speedup stats:")
print(f"  Min: {df_htl['speedup'].min():.2f}x")
print(f"  P25: {df_htl['speedup'].quantile(0.25):.2f}x")
print(f"  Median: {df_htl['speedup'].median():.2f}x")
print(f"  P75: {df_htl['speedup'].quantile(0.75):.2f}x")
print(f"  Max: {df_htl['speedup'].max():.2f}x")
print(f"  % faster than MILP: {(df_htl['speedup'] > 1).mean()*100:.0f}%")

In [ ]:
# --- Cost Gap Analysis: High-Crit MaxTimeLimit ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel A: Cost gap distribution
ax = axes[0]
ax.hist(df_htl['cost_gap_pct'], bins=25, color='#e74c3c', alpha=0.7, edgecolor='white')
ax.axvline(x=0, color='green', linestyle='--', lw=2, label='No gap')
ax.axvline(x=df_htl['cost_gap_pct'].median(), color='blue', linestyle='--', lw=2,
           label=f"Median={df_htl['cost_gap_pct'].median():.1f}%")
ax.set_xlabel('Cost Gap vs MILP (%)')
ax.set_ylabel('Count')
ax.set_title('(A) Cost Gap Distribution')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Panel B: Cost gap vs n_zones
ax = axes[1]
sc = ax.scatter(df_htl['n_zones'], df_htl['cost_gap_pct'],
                c=df_htl['speedup'], cmap='RdYlGn', s=60, alpha=0.8,
                edgecolors='white', linewidth=0.5)
ax.axhline(y=0, color='green', linestyle='--', alpha=0.5)
ax.set_xlabel('Number of Zones')
ax.set_ylabel('Cost Gap (%)')
ax.set_title('(B) Cost Gap vs Problem Size')
plt.colorbar(sc, ax=ax, label='Speedup (x)')
ax.grid(True, alpha=0.3)

# Panel C: Pipeline obj vs MILP obj (scatter)
ax = axes[2]
ax.scatter(df_htl['milp_objective'] / 1e6, df_htl['pipeline_objective'] / 1e6,
           c='#e74c3c', s=60, alpha=0.7, edgecolors='white', linewidth=0.5)
obj_range = [min(df_htl['milp_objective'].min(), df_htl['pipeline_objective'].min()) / 1e6,
             max(df_htl['milp_objective'].max(), df_htl['pipeline_objective'].max()) / 1e6]
ax.plot(obj_range, obj_range, 'k--', alpha=0.3, label='x = y')
ax.set_xlabel('MILP Objective (M EUR)')
ax.set_ylabel('Pipeline Objective (M EUR)')
ax.set_title('(C) Objective Comparison')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

fig.suptitle('Cost Gap Analysis: High-Criticality MaxTimeLimit Scenarios', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig10_htl_cost_gap.png', dpi=300, bbox_inches='tight')
plt.show()

# Print stats
print(f"Cost gap stats (pipeline vs MILP best incumbent):")
print(f"  Mean: {df_htl['cost_gap_pct'].mean():.1f}%")
print(f"  Median: {df_htl['cost_gap_pct'].median():.1f}%")
print(f"  P10: {df_htl['cost_gap_pct'].quantile(0.10):.1f}%")
print(f"  P90: {df_htl['cost_gap_pct'].quantile(0.90):.1f}%")
print(f"  Pipeline better (gap < 0): {(df_htl['cost_gap_pct'] < 0).sum()}/{len(df_htl)} "
      f"({(df_htl['cost_gap_pct'] < 0).mean()*100:.0f}%)")
print(f"  Within 5%: {(df_htl['cost_gap_pct'].abs() < 5).sum()}/{len(df_htl)}")
print(f"  Within 10%: {(df_htl['cost_gap_pct'].abs() < 10).sum()}/{len(df_htl)}")

# Stage distribution for this subset
print(f"\nLP Stage distribution:")
for stage, count in df_htl['pipeline_stage'].value_counts().items():
    print(f"  {stage}: {count} ({count/len(df_htl)*100:.0f}%)")

In [ ]:
# --- Economic Advantage: High-Crit MaxTimeLimit ---
from src.eval.economic_advantage import EconomicAdvantageAnalyzer

econ_htl = EconomicAdvantageAnalyzer(df_htl)

# Sensitivity analysis
lambda_values = [1, 5, 10, 20, 50, 100]
sens_htl = econ_htl.sensitivity_analysis(lambda_values=lambda_values)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel A: % Profitable vs Lambda
ax = axes[0]
ax.plot(sens_htl['lambda'], sens_htl['pct_profitable'], 'b-o', lw=2, markersize=5)
ax.axhline(y=50, color='red', linestyle=':', alpha=0.5, label='50%')
ax.axhline(y=80, color='orange', linestyle=':', alpha=0.5, label='80%')
ax.axhline(y=95, color='green', linestyle=':', alpha=0.5, label='95%')
ax.set_xlabel('Lambda (EUR/second)', fontweight='bold')
ax.set_ylabel('% Scenarios Pipeline Profitable', fontweight='bold')
ax.set_title('(A) Profitability vs Time Value')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 105)
ax.set_xscale('log')

# Panel B: Mean Advantage vs Lambda
ax = axes[1]
ax.plot(sens_htl['lambda'], sens_htl['mean_advantage'] / 1e6, 'g-o', lw=2, markersize=5)
ax.axhline(y=0, color='red', linestyle='--', alpha=0.5)
ax.set_xlabel('Lambda (EUR/second)', fontweight='bold')
ax.set_ylabel('Mean Advantage (M EUR)', fontweight='bold')
ax.set_title('(B) Mean Economic Advantage')
ax.grid(True, alpha=0.3)
ax.set_xscale('log')

# Panel C: Advantage decomposition at selected lambda
ax = axes[2]
selected_lambdas = [1, 5, 10, 20, 50, 100]
for lam in selected_lambdas:
    adv_df = econ_htl.compute_advantage(lam)
    valid = adv_df.dropna(subset=['advantage'])
    time_component = valid['advantage_from_time'].mean() / 1e6
    cost_component = valid['advantage_from_cost'].mean() / 1e6
    ax.bar(f'{lam}', time_component, color='#2ecc71', alpha=0.8, label='Time saved' if lam == selected_lambdas[0] else '')
    ax.bar(f'{lam}', cost_component, bottom=time_component, color='#e74c3c', alpha=0.8,
           label='Cost gap' if lam == selected_lambdas[0] else '')

ax.axhline(y=0, color='black', linestyle='-', lw=0.5)
ax.set_xlabel('Lambda (EUR/s)', fontweight='bold')
ax.set_ylabel('Mean Advantage (M EUR)', fontweight='bold')
ax.set_title('(C) Advantage Decomposition')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

fig.suptitle('Economic Advantage: High-Criticality MaxTimeLimit Scenarios', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig11_htl_economic_advantage.png', dpi=300, bbox_inches='tight')
plt.show()

# Breakeven analysis
summary_htl = econ_htl.compute_summary(lambda_values=[1, 5, 10, 20, 50, 100])
print('=' * 70)
print(f'ECONOMIC ADVANTAGE: High-Crit MaxTimeLimit (n={len(df_htl)})')
print('=' * 70)
print(f"Breakeven lambda (50% profitable): {summary_htl['breakeven_lambda_50pct']:.2f} EUR/s")
print(f"Breakeven lambda (80% profitable): {summary_htl['breakeven_lambda_80pct']:.2f} EUR/s")
print(f"Breakeven lambda (95% profitable): {summary_htl['breakeven_lambda_95pct']:.2f} EUR/s")
print(f"\nAt selected lambda values:")
for key, val in sorted(summary_htl.items()):
    if key.startswith('lambda_'):
        lam = key.replace('lambda_', '')
        print(f"  lambda={lam:>4s} EUR/s: {val['pct_profitable']:5.1f}% profitable, "
              f"mean advantage={val['mean_advantage_eur']/1e6:+.2f} MEUR")

In [ ]:
# --- Summary Table: High-Crit MaxTimeLimit vs All High-Crit ---
df_high_all = df[df['family'] == 'high']
df_high_opt = df_high_all[df_high_all['milp_termination'] == 'optimal']
PIPE_STAGE_COL = 'pipeline_stage_reached' if 'pipeline_stage_reached' in df.columns else 'pipeline_stage'

print('=' * 90)
print('HIGH-CRITICALITY: MaxTimeLimit vs Optimal vs All')
print('=' * 90)

for label, sub in [('MaxTimeLimit', df_htl), ('Optimal', df_high_opt), ('All High', df_high_all)]:
    valid = sub.dropna(subset=['speedup', 'cost_gap_pct'])
    n_better = (valid['cost_gap_pct'] < 0).sum()
    print(f"\n  {label} (n={len(sub)}):")
    print(f"    Pipeline time:  mean={sub['pipeline_solve_time'].mean():.1f}s, median={sub['pipeline_solve_time'].median():.1f}s")
    print(f"    MILP time:      mean={sub['milp_solve_time'].mean():.1f}s, median={sub['milp_solve_time'].median():.1f}s")
    print(f"    Speedup:        mean={valid['speedup'].mean():.1f}x, median={valid['speedup'].median():.1f}x")
    print(f"    Cost gap:       P50={valid['cost_gap_pct'].median():.1f}%, P90={np.percentile(valid['cost_gap_pct'], 90):.1f}%")
    print(f"    Pipeline better: {n_better}/{len(valid)} ({n_better/max(len(valid),1)*100:.0f}%)")
    print(f"    Deepest stage:  {dict(sub[PIPE_STAGE_COL].value_counts())}")

# Save HTL subset
df_htl.to_csv(OUTPUT_DIR / 'high_crit_timelimit_comparison.csv', index=False)
print(f"\nSaved HTL subset to {OUTPUT_DIR / 'high_crit_timelimit_comparison.csv'}")
